##### UNHCR Refugee Population Statistics Database
#### Asylum Application and Decision Data
##### Worldwide from 2000 onwards

All data comes from the UNHCR's Public API. Countries submit returns to the UNHCR on the volumes of asylum
application and decision data annually. This includes decision volumes by type, such as whether applications were
granted or refused. Additional decision outcomes are 'Other protection' which refers to humanitarian protection or
special grants inplace of regular asylum, and 'Otherwise closed' which refers to the closing of applications
prior to an asylum decision being made. For
more
information, see:
https://www.unhcr.org/refugee-statistics/

Datasets terms of use: https://www.unhcr.org/what-we-do/data-and-publications/data-and-statistics/terms-use-datasets

The aim of this analysis is to explore asylum application and decision data from the United Nations High Commissioner
for Refugees
(UNHCR) and to present key findings and insights. A secondary aim is to demonstrate the use of Python for data
analysis and visualisation. This analysis was undertaken independently of the UNHCR using publicly-available data.

#### A. PRE-PROCESSING

PRE-PROCESSING - IMPORT PYTHON PACKAGE REQUIREMENTS

In [ ]:
# %% Import Python Package Requirements
from io import BytesIO
from datetime import date
from zipfile import BadZipFile, ZipFile
from datetime import datetime
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import requests
from matplotlib import style
import matplotlib.ticker as mtick
from matplotlib.ticker import StrMethodFormatter, EngFormatter
from IPython.display import display
import pycountry
import ipywidgets as widgets
from IPython.display import clear_output, display
import numpy as np
import datetime
import os
from pathlib import Path

mpl.rcParams.update(mpl.rcParamsDefault)
mpl.rcParams.update({
    'font.weight': 'normal',
    'axes.titleweight': 'normal',
    'axes.labelweight': 'normal',
    'figure.titleweight': 'normal',
})

def _normalise_figure_font_weights(figure):
    """Use regular-weight text consistently in every chart."""
    for text_artist in figure.findobj(match=mpl.text.Text):
        text_artist.set_fontweight('normal')

if not hasattr(plt, '_unhcr_original_show'):
    plt._unhcr_original_show = plt.show

def _unhcr_regular_weight_show(*args, **kwargs):
    for figure_number in plt.get_fignums():
        _normalise_figure_font_weights(plt.figure(figure_number))
    return plt._unhcr_original_show(*args, **kwargs)

plt.show = _unhcr_regular_weight_show

PRE-PROCESSING - PYTHON FUNCTIONS FOR FORMATTING DATA

In [ ]:
# %% Formatting Functions for Visualisations
# Function to calculate percentages for pie charts
def format_pie_slice(pct, allvalues):
    """Function to add percentage labels to pie charts """
    absolute = int(pct / 100. * np.sum(allvalues))
    return "{:.1f}%".format(pct, absolute)

# Function to format application outcome labels
def format_application_outcome_label(
        applications, grant_rate, other_protection_rate, refused_rate,
        otherwise_closed_rate):
    """Function to format an accessible application-volume label with all decision
    rates."""
    def rate_label(rate):
        return f'{rate:.0%}' if pd.notna(rate) else 'n/a'

    return (
        f'{format_volume_label(applications)}\n'
        f'G {rate_label(grant_rate)} | OP {rate_label(other_protection_rate)} '
        f'| '
        f'R {rate_label(refused_rate)
        } | C {rate_label(otherwise_closed_rate)}'
    )

# Function to format volume labels
def format_volume_label(value):
    if value >= 1_000_000:
        return f'{value / 1_000_000:.1f}m'
    if value >= 1_000:
        return f'{value / 1_000:.0f}k'
    return f'{value:,.0f}'

Pre-processing - Function to download UNHCR Asylum Application and Decision Data

In [ ]:
# %% UNHCR API Parameters
API_BASE_URL = "https://api.unhcr.org/population/v1"
YEAR_FROM = 1993
last_calendar_year = date.today().year - 1
REQUEST_TIMEOUT_SECONDS = 120
# Default configuration constants (adjust as needed)
CACHE_DIR = Path("./unhcr_cache")
# The UNHCR footnotes identify only these asylum-country/year observations as
# applications reported in cases rather than persons.  A multiplier may be used
# only after aggregating by country of asylum; UNHCR advises against
# applying it
# to country-of-origin results because average case size varies by origin.
# USA 2025 is retained as an explicit 1.0 multiplier because UNHCR counts one
# person per case for the relevant USCIS data that year.
APPLICATION_CASE_TO_PERSON_MULTIPLIERS = {
    **{("GBR", year): 1.3 for year in range(2016, 2023)},
    ("USA", 2016): 1.46,
    ("USA", 2018): 1.501,
    ("USA", 2019): 1.501,
    ("USA", 2020): 1.5,
    ("USA", 2025): 1.0,
    ("KAZ", 2018): 3.0,
}
# %% Pre-processing - Function to download UNHCR CSV data
def get_latest_release_date(current_dt: datetime.datetime = None) -> datetime.datetime:
    """
    Returns the cutoff date of the most recent UNHCR data release cycle.
    - Releases occur in June (effective July 1) and December (effective January 1).
    """
    if current_dt is None:
        current_dt = datetime.datetime.now(datetime.timezone.utc)

    year = current_dt.year
    july_release = datetime.datetime(year, 7, 1, tzinfo=datetime.timezone.utc)
    jan_release = datetime.datetime(year, 1, 1, tzinfo=datetime.timezone.utc)

    if current_dt >= july_release:
        return july_release
    elif current_dt >= jan_release:
        return jan_release
    else:
        # Before Jan 1, the latest release was July 1 of the prior year
        return datetime.datetime(year - 1, 7, 1, tzinfo=datetime.timezone.utc)

def download_unhcr_csv(
    dataset: str,
    cache_dir: Path = CACHE_DIR,
    force_refresh: bool = False,
) -> pd.DataFrame:
    """
    Download or load cached UNHCR dataset with semi-annual cache invalidation.
    """
    cache_dir = Path(cache_dir)
    cache_dir.mkdir(parents=True, exist_ok=True)
    cache_file = cache_dir / f"{dataset}_from_{YEAR_FROM}.parquet"

    latest_release = get_latest_release_date()

    # Check if cache exists and is still valid (downloaded after the latest release)
    if cache_file.exists() and not force_refresh:
        mtime = datetime.datetime.fromtimestamp(
            cache_file.stat().st_mtime, tz=datetime.timezone.utc
        )
        if mtime >= latest_release:
            df = pd.read_parquet(cache_file)
            df.attrs["dataset_name"] = dataset
            df.attrs["downloaded_at"] = mtime.strftime("%Y-%m-%d %H:%M:%S UTC")
            df.attrs["cache_source"] = "local_cache"
            return df

    # Fetch from UNHCR API if cache is missing or stale
    download_time = datetime.datetime.now(datetime.timezone.utc)
    response = requests.get(
        f"{API_BASE_URL}/{dataset}/",
        params={
            "yearFrom": YEAR_FROM,
            "download": "true",
            "coo_all": "true",
            "coa_all": "true",
            "cf_type": "ISO",
        },
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    response.raise_for_status()

    try:
        with ZipFile(BytesIO(response.content)) as archive:
            csv_name = f"{dataset}.csv"
            matched_name = next(
                (f for f in archive.namelist() if f.endswith(csv_name)), None
            )
            if not matched_name:
                raise ValueError(
                    f"Expected {csv_name} in download, found {archive.namelist()}"
                )

            with archive.open(matched_name) as csv_file:
                df = pd.read_csv(csv_file)

            # Save to cache
            df.to_parquet(cache_file, index=False)

            # Metadata attributes
            df.attrs["dataset_name"] = dataset
            df.attrs["downloaded_at"] = download_time.strftime("%Y-%m-%d %H:%M:%S UTC")
            df.attrs["cache_source"] = "api_download"
            return df

    except BadZipFile as error:
        raise RuntimeError(
            f"UNHCR did not return a valid ZIP archive for {dataset}."
        ) from error

In [ ]:
# %% Data Refresh Controls, Cache, and Provenance
# Set to True for a deliberate refresh from both source APIs.
FORCE_DATA_REFRESH = False
EUROSTAT_CACHE_MAX_AGE_DAYS = 14

OBSERVED_DATA_CACHE_DIR = CACHE_DIR / 'observed_stock_cache'
OBSERVED_DATA_SNAPSHOT_DIR = OBSERVED_DATA_CACHE_DIR / 'snapshots'
OBSERVED_DATA_CACHE_DIR.mkdir(parents=True, exist_ok=True)
OBSERVED_DATA_SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)

def cache_is_fresh(
        cache_file,
        release_cutoff=None,
        max_age_days=None,
):
    """Return whether a cache file passes its release/age refresh policy."""
    cache_file = Path(cache_file)
    if not cache_file.exists() or FORCE_DATA_REFRESH:
        return False

    modified_at = datetime.datetime.fromtimestamp(
        cache_file.stat().st_mtime,
        tz=datetime.timezone.utc,
    )
    if release_cutoff is not None and modified_at < release_cutoff:
        return False
    if max_age_days is not None:
        return (
            datetime.datetime.now(datetime.timezone.utc) - modified_at
        ).days < max_age_days
    return True

def save_refresh_metadata(metadata_file, metadata):
    """Persist a compact, human-readable record of the current extract."""
    pd.Series(metadata).to_json(
        metadata_file,
        indent=2,
        date_format='iso',
    )

def save_snapshot(data, dataset_name, retrieved_at):
    """Retain a dated extract so published charts can be reproduced later."""
    snapshot_stamp = retrieved_at.strftime('%Y%m%dT%H%M%SZ')
    snapshot_file = (
        OBSERVED_DATA_SNAPSHOT_DIR
        / f'{dataset_name}_{snapshot_stamp}.parquet'
    )
    data.to_parquet(snapshot_file, index=False)
    return snapshot_file

In [ ]:
# %% Latest UNHCR Data Release
latest_release_date = get_latest_release_date()
print(f"Latest UNHCR data release: {latest_release_date:%d %B %Y}")

PRE-PROCESSING - DOWNLOAD UNHCR ASYLUM APPLICATIONS DATA

In [ ]:
# %% Download UNHCR Asylum Applications
# Use force_refresh=False to avoid downloading the data again if it's already cached
UNapps = download_unhcr_csv("asylum-applications", force_refresh=False)
first_application_year = int(UNapps['Year'].min())
latest_application_year = int(UNapps['Year'].max())
# Access the stored metadata
print("Source:", UNapps.attrs.get("cache_source"))
print("Dataset:", UNapps.attrs.get("dataset_name"))
print("Downloaded at:", UNapps.attrs.get("downloaded_at"))
print(f'First Application Year: {first_application_year}')
print(f'Latest Application Year: {latest_application_year}')

PRE-PROCESSING - DOWNLOAD UNHCR ASYLUM DECISIONS DATA

In [ ]:
# %% Download UNHCR Decisions
# Use force_refresh=False to avoid downloading the data again if it's already cached
UNdecs = download_unhcr_csv("asylum-decisions", force_refresh=False)
first_decision_year = int(UNdecs['Year'].min())
latest_decision_year = int(UNdecs['Year'].max())
# Access the stored metadata
print("Source:", UNdecs.attrs.get("cache_source"))
print("Dataset:", UNdecs.attrs.get("dataset_name"))
print("Downloaded at:", UNdecs.attrs.get("downloaded_at"))
print(f'First Decision Year: {first_decision_year}')
print(f'Latest Decision Year: {latest_decision_year}')

In [ ]:
# %% Download UNHCR Pending Historical Data
DATA_DIR = Path("data")
PENDING_HISTORICAL_FILE = Path('persons_of_concern.csv')

if not PENDING_HISTORICAL_FILE.exists():
    raise FileNotFoundError(
        f"Missing {PENDING_HISTORICAL_FILE}. "
        "Place persons_of_concern.csv in the data folder."
    )

In [ ]:
# %% Calculate Pending Historical Applications
UNpendingHistorical = (
    pd.read_csv("persons_of_concern.csv")
    .rename(
        columns={
            "Country of Asylum": "Country of asylum",
            "Country of Origin": "Country of origin",
            "Country of Asylum ISO": "Country of asylum (ISO)",
            "Country of Origin ISO": "Country of origin (ISO)",
            "Asylum-seekers": "Pending asylum-seekers",
        }
    )
)

UNpendingHistorical["Pending asylum-seekers"] = pd.to_numeric(
    UNpendingHistorical["Pending asylum-seekers"],
    errors="coerce",
).fillna(0)

In [ ]:
# %% Pending Asylum Applicant Stock Prior to 2000

opening_pending_2000 = (
    UNpendingHistorical.loc[
        UNpendingHistorical["Year"].eq(1999),
        "Pending asylum-seekers",
    ]
    .sum()
)

print(
    f"Worldwide Pending Asylum Applicant Stock at 1 January 2000: "
    f"{opening_pending_2000:,.0f}"
)

PRE-PROCESSING - ESTABLISH THE LATEST YEAR OF DATA AVAILABLE

Show the first and latest available year of data

In [ ]:
# %% First Applications Year
f'The first Applications year is: {first_application_year}'

In [ ]:
# %% Latest Decisions Year
f'The latest Decisions year is: {latest_decision_year}'

PRE-PROCESSING - BUILD ISO COUNTRY CODE LOOKUP TABLE

This table is built using the Python 'pycountry' module instead
of UNHCR's data so that an 'Unknown' value and code ('UNK') can be added. The Unknown value and UNK code are used
against data where the country of origin is unknown. This can sometimes occur when participating countries have not
yet sent a full return to the UNHCR. It can also occur in historic records where the country of origin was not
recorded or records were unavailable.

In [ ]:
# %% Prepare ISO Country Code Lookup Table
# Uses pycountry module instead of UNHCR's data so that an 'Unknown' value can be added.

iso_country_names = pd.DataFrame(
    [
        {
            'ISO country code': country.alpha_3,
            'Country name': country.name,
        }
        for country in pycountry.countries
    ]
)

iso_country_names = pd.concat(
    [
        iso_country_names,
        pd.DataFrame(
            {
                'ISO country code': ['UNK'],
                'Country name': ['Unknown'],
            }
        ),
    ],
    ignore_index=True,
)
iso_country_names_display = (
    iso_country_names
    .sort_values('Country name')
    .reset_index(drop=True)
)
# %%
country_names_by_iso = iso_country_names.set_index(
    'ISO country code'
)['Country name']

def country_label(country_code):
    return country_names_by_iso.get(country_code, country_code)

apps_lookup_source = (
    UNapps.reset_index()
    if isinstance(UNapps.index, pd.MultiIndex)
    else UNapps.copy()
)

decs_lookup_source = (
    UNdecs.reset_index()
    if isinstance(UNdecs.index, pd.MultiIndex)
    else UNdecs.copy()
)

applications_coverage = (
    apps_lookup_source
    .groupby("Country of asylum (ISO)")["Year"]
    .agg(["min", "max"])
    .reset_index()
    .rename(
        columns={
            "Country of asylum (ISO)": "ISO country code",
            "min": "Applications first year",
            "max": "Applications last year",
        }
    )
)

decisions_coverage = (
    decs_lookup_source
    .groupby("Country of asylum (ISO)")["Year"]
    .agg(["min", "max"])
    .reset_index()
    .rename(
        columns={
            "Country of asylum (ISO)": "ISO country code",
            "min": "Decisions first year",
            "max": "Decisions last year",
        }
    )
)

historical_pending_coverage = (
    UNpendingHistorical
    .groupby("Country of asylum (ISO)", as_index=False)
    .agg(
        **{
            "Historical asylum-seeker entries": ("Year", "nunique"),
            "Historical non-zero years": (
                "Pending asylum-seekers",
                lambda values: values.gt(0).sum(),
            ),
        }
    )
    .rename(
        columns={
            "Country of asylum (ISO)": "ISO country code",
        }
    )
)

iso_country_names_display = (
    iso_country_names
    .merge(applications_coverage, on="ISO country code", how="left")
    .merge(decisions_coverage, on="ISO country code", how="left")
    .merge(historical_pending_coverage, on="ISO country code", how="left")
)

for column in [
    "Applications first year",
    "Applications last year",
    "Decisions first year",
    "Decisions last year",
    "Historical asylum-seeker entries",
    "Historical non-zero years",
]:
    iso_country_names_display[column] = (
        iso_country_names_display[column]
        .fillna(0)
        .astype(int)
    )

iso_country_names_display["UNHCR asylum data held"] = np.select(
    [
        (
            iso_country_names_display["Applications first year"].gt(0)
            & iso_country_names_display["Decisions first year"].gt(0)
        ),
        iso_country_names_display["Applications first year"].gt(0),
        iso_country_names_display["Decisions first year"].gt(0),
    ],
    [
        "Applications and decisions",
        "Applications only",
        "Decisions only",
    ],
    default="No applications or decisions",
)

iso_country_names_display = (
    iso_country_names_display
    .sort_values(["UNHCR asylum data held", "Country name"])
    .reset_index(drop=True)
)
iso_country_names_display["Country / aggregate status"] = np.where(
    iso_country_names_display["ISO country code"].eq("UNK"),
    "Unknown or unallocated UNHCR aggregate",
    "Named country",
)

Show the ISO Country Code Lookup Table

In [ ]:
# %% Display ISO Country Code Lookup Table
page_size = 25
total_pages = int(np.ceil(len(iso_country_names_display) / page_size))
state = {'page': 1}

previous_button = widgets.Button(description='← Previous')
next_button = widgets.Button(description='Next →')
page_label = widgets.Label()
table_output = widgets.Output()

def show_page():
    page_number = state['page']
    start_row = (page_number - 1) * page_size
    end_row = start_row + page_size

    page_label.value = f'Page {page_number} of {total_pages}'
    previous_button.disabled = page_number == 1
    next_button.disabled = page_number == total_pages

    with table_output:
        clear_output(wait=True)
        display(
            iso_country_names_display
            .iloc[start_row:end_row]
            .style
            .hide(axis='index')
        )

def previous_page(_):
    state['page'] = max(1, state['page'] - 1)
    show_page()

def next_page(_):
    state['page'] = min(total_pages, state['page'] + 1)
    show_page()

previous_button.on_click(previous_page)
next_button.on_click(next_page)

display(widgets.HBox([previous_button, page_label, next_button]))
display(table_output)

show_page()

Pre-Processing - Group Applications and Decisions by Year, Origin, and Receiving Country

In [ ]:
# %% Group Applications and Decisions
# Group applications and decisions by year, origin and asylum country.
# Case-to-person conversions are deliberately not applied here: UNHCR permits
# them only when data are summarised by country of asylum, not by origin.

# Group applications, retaining the UNHCR application type code:
# N = New, R = Repeat, A = Appeal (with other source-specific codes retained).
unit_column = "Cases / Persons"

# Applications: retain application type and reporting unit.
UNapps = (
    UNapps.groupby(
        [
            "Year",
            "Country of origin (ISO)",
            "Country of asylum (ISO)",
            "Application type",
            unit_column,
        ]
    )
    .agg(applied=("applied", "sum"))
)

# Decisions: retain reporting unit.
UNdecs = (
    UNdecs.groupby(
        [
            "Year",
            "Country of origin (ISO)",
            "Country of asylum (ISO)",
            unit_column,
        ]
    )
    .agg(
        **{
            "Recognized decisions": ("Recognized decisions", "sum"),
            "Complementary protection": ("Complementary protection", "sum"),
            "Rejected decisions": ("Rejected decisions", "sum"),
            "Otherwise closed": ("Otherwise closed", "sum"),
            "Total decisions": ("Total decisions", "sum"),
        }
    )
)

Pre-processing - Calculate Grant, Complementary or Other Protection Rates, Rejected Decisions and Otherwise Closed Rates

In [ ]:
# %% Calculate Grant Rates
decision_total = UNdecs["Total decisions"].replace(0, np.nan)

UNdecs["Granted"] = (
    UNdecs["Recognized decisions"] / decision_total
)
UNdecs["Other protection"] = (
    UNdecs["Complementary protection"] / decision_total
)
UNdecs["Refused"] = (
    UNdecs["Rejected decisions"] / decision_total
)
UNdecs["Otherwise Closed"] = (
    UNdecs["Otherwise closed"] / decision_total
)

rate_columns = [
    "Granted",
    "Other protection",
    "Refused",
    "Otherwise Closed",
]

# Require all four outcome rates before validating their total.
UNdecs["Total%"] = UNdecs[rate_columns].sum(
    axis=1,
    min_count=len(rate_columns),
)

UNdecsCheckTotal = UNdecs.loc[
    UNdecs["Total decisions"].gt(0)
    & UNdecs["Total%"].notna()
    & ~np.isclose(UNdecs["Total%"], 1.0)
].copy()

print("Decision rows:", len(UNdecs))
print("Rows whose rates do not total 100%:", len(UNdecsCheckTotal))

UNdecs.drop(columns="Total%", inplace=True)

Pre-processing - Join Grouped Application and Decision Datasets

In [ ]:
# %% Join Apps and Decs Datasets

join_keys = [
    "Year",
    "Country of origin (ISO)",
    "Country of asylum (ISO)",
    "Cases / Persons",
]

# Works whether the prior cells left a MultiIndex or normal columns.
apps_detail = UNapps.reset_index() if isinstance(UNapps.index, pd.MultiIndex) else UNapps.copy()
decs_detail = UNdecs.reset_index() if isinstance(UNdecs.index, pd.MultiIndex) else UNdecs.copy()

# Sum application types before joining, so decisions are not duplicated.
apps_for_decisions = (
    apps_detail
    .groupby(join_keys, as_index=False)["applied"]
    .sum()
)

# Each decision record must have one row per joining combination.
if decs_detail.duplicated(join_keys).any():
    raise ValueError(
        "UNdecs contains duplicate decision rows. Rerun the decision-grouping cell."
    )

UNTotalAppsDecs = apps_for_decisions.merge(
    decs_detail,
    how="left",
    on=join_keys,
    validate="one_to_one",
)

# Keep the detailed tables for later analysis.
UNapps = apps_detail
UNdecs = decs_detail

Pre-Processing - Create Receiving Country Dataset (removes country of origin)

In [ ]:
# %% Calculate Estimated Person Volumes using Case Multipliers
FALLBACK_CASE_TO_PERSON_MULTIPLIER = np.mean(
    list(APPLICATION_CASE_TO_PERSON_MULTIPLIERS.values())
)

print(
    "Fallback estimated persons-per-case multiplier:",
    round(FALLBACK_CASE_TO_PERSON_MULTIPLIER, 2),
)

def estimate_persons_from_cases(dataframe, value_columns):
    result = dataframe.copy()

    record_keys = pd.Series(
        list(
            zip(
                result["Country of asylum (ISO)"],
                result["Year"],
            )
        ),
        index=result.index,
    )

    documented_multiplier = record_keys.map(
        APPLICATION_CASE_TO_PERSON_MULTIPLIERS
    )

    case_rows = result["Cases / Persons"].eq("C")

    result["Person-estimation multiplier"] = 1.0
    result.loc[case_rows, "Person-estimation multiplier"] = (
        documented_multiplier.loc[case_rows]
        .fillna(FALLBACK_CASE_TO_PERSON_MULTIPLIER)
    )

    result["Person-estimation basis"] = np.select(
        [
            ~case_rows,
            case_rows & documented_multiplier.notna(),
            case_rows & documented_multiplier.isna(),
        ],
        [
            "Reported persons",
            "Estimated persons — documented multiplier",
            "Estimated persons — average multiplier",
        ],
        default="Unknown",
    )

    for column in value_columns:
        result[f"{column} (estimated persons)"] = (
            result[column] * result["Person-estimation multiplier"]
        )

    return result


UNappsEstimated = estimate_persons_from_cases(
    UNapps,
    ["applied"],
)

UNdecsEstimated = estimate_persons_from_cases(
    UNdecs,
    [
        "Recognized decisions",
        "Complementary protection",
        "Rejected decisions",
        "Otherwise closed",
        "Total decisions",
    ],
)


In [ ]:
# %% Calculate Application Totals - Persons and Cases Combined
''' This cell builds a country-level ranking of asylum applications from 2000 onward.
For each receiving country, it:
 - totals applications reported as people (P);
 - totals applications reported as cases (C);
 - adds those two values into a mixed-unit total;
 - calculates the proportion reported as people versus cases;
 - selects the 20 countries with the largest mixed-unit totals.
 - applications_by_country_unit contains the full country table.
 - top_20_worldwide_countries contains only the top 20.
The important caveat is that the ranking combines people and cases, which are different units. It is useful as a descriptive way to identify large reporting countries, but not as a true people-based volume comparison.'''

applications_by_country_unit = (
    UNTotalAppsDecs.loc[
        UNTotalAppsDecs["Year"].between(2000, latest_available_year)
    ]
    .groupby(
        ["Country of asylum (ISO)", "Cases / Persons"],
        as_index=False,
    )["applied"]
    .sum()
    .pivot(
        index="Country of asylum (ISO)",
        columns="Cases / Persons",
        values="applied",
    )
    .fillna(0)
    .rename(columns={
        "P": "Applications reported as persons",
        "C": "Applications reported as cases",
    })
)

applications_by_country_unit["Applications (mixed reporting units)"] = (
    applications_by_country_unit["Applications reported as persons"]
    + applications_by_country_unit["Applications reported as cases"]
)

mixed_total = applications_by_country_unit[
    "Applications (mixed reporting units)"
].replace(0, np.nan)

applications_by_country_unit["Person-reported share"] = (
    applications_by_country_unit["Applications reported as persons"] / mixed_total
)

applications_by_country_unit["Case-reported share"] = (
    applications_by_country_unit["Applications reported as cases"] / mixed_total
)

top_20_worldwide_countries = (
    applications_by_country_unit
    .reset_index()
    .nlargest(20, "Applications (mixed reporting units)")
)
# %% Persons and Cases Display by Receiving Country
applications_by_country_and_unit = (
    applications_by_country_unit[
        [
            "Applications reported as persons",
            "Applications reported as cases",
        ]
    ]
    .rename(
        columns={
            "Applications reported as persons": "Applications (persons)",
            "Applications reported as cases": "Applications (cases)",
        }
    )
    .reset_index()
)

In [ ]:
# %% Create Receiving Country Dataset

''' This creates a receiving-country dataset from UNTotalAppsDecs. It:
 - groups records by year, asylum country, and reporting unit (P persons or C cases);
 - sums applications and all decision-outcome volumes within each group;
 - deliberately keeps persons and cases separate, avoiding invalid unit mixing;
 - copies applied into a clearer Applications (reported) column;
 - adds a readable reporting-unit label;
 - calculates Decisions minus applications for the same year and same reporting unit;
 - calculates FVP, the ratio of total decisions to applications. FVP is not a grant rate or pending backlog measure.
 For example, FVP = 1.2 means the reported decision volume was 20% greater than reported applications in that country/year/unit.'''

volume_columns = [
    "applied",
    "Recognized decisions",
    "Complementary protection",
    "Rejected decisions",
    "Otherwise closed",
    "Total decisions",
]

UNTotalAppsDecs = (
    UNTotalAppsDecs
    .groupby(
        [
            "Year",
            "Country of asylum (ISO)",
            "Cases / Persons",
        ],
        as_index=False,
    )[volume_columns]
    .sum()
)

# Preserve the source's reporting unit and unconverted application total.
UNTotalAppsDecs["Applications (reported)"] = UNTotalAppsDecs["applied"]

UNTotalAppsDecs["Reporting unit"] = (
    UNTotalAppsDecs["Cases / Persons"]
    .map({
        "P": "Persons (reported)",
        "C": "Cases (reported)",
    })
    .fillna("Unknown reporting unit")
)

# This is a same-unit comparison, not a pending-case backlog.
UNTotalAppsDecs["Decisions minus applications"] = (
    UNTotalAppsDecs["Total decisions"]
    - UNTotalAppsDecs["applied"]
)

# Retained as FVP if later cells use the name.
UNTotalAppsDecs["FVP"] = (
    UNTotalAppsDecs["Total decisions"]
    / UNTotalAppsDecs["applied"].replace(0, np.nan)
)

Worldwide 2000 Onwards - Show the Top 20 Receiving Countries

In [ ]:
# %% Top 20 Countries since 2000 - Person and Case Application Volumes
# %% Displays summary statistics for the Top 20 Receiving Countries by Application Volume since 2000

top_20_worldwide_display = (
    top_20_worldwide_countries
    .rename(columns={"Country of asylum (ISO)": "ISO country code"})
    .merge(
        iso_country_names.set_index("ISO country code")[["Country name"]],
        left_on="ISO country code",
        right_index=True,
        how="left",
    )
    [
        [
            "ISO country code",
            "Country name",
            "Applications (mixed reporting units)",
            "Applications reported as persons",
            "Applications reported as cases",
            "Person-reported share",
            "Case-reported share",
        ]
    ]
    .sort_values("Applications (mixed reporting units)", ascending=False)
    .reset_index(drop=True)
)

top_20_worldwide_display.style.format(
    {
        "Applications (mixed reporting units)": "{:,.0f}",
        "Applications reported as persons": "{:,.0f}",
        "Applications reported as cases": "{:,.0f}",
        "Person-reported share": "{:.1%}",
        "Case-reported share": "{:.1%}",
    }
).hide(axis="index")


Worldwide 2000 Onwards - Plots

In [ ]:
# %% Person and Case Totals for Top 20 Countries Since 2000 and All Other Countries

worldwide_data = UNTotalAppsDecs.loc[
    UNTotalAppsDecs["Year"].between(2000, latest_available_year)
].copy()

top_20_countries = top_20_worldwide_countries[
    "Country of asylum (ISO)"
]

# Worldwide totals by reporting unit.
worldwide_totals_by_unit = (
    worldwide_data
    .groupby("Cases / Persons")["applied"]
    .sum()
    .reindex(["P", "C"], fill_value=0)
)

TotalAppsPersons = worldwide_totals_by_unit["P"]
TotalAppsCases = worldwide_totals_by_unit["C"]
TotalAppsMixed = TotalAppsPersons + TotalAppsCases

# Top-20 totals by reporting unit.
TotalAppsTop20 = worldwide_data.loc[
    worldwide_data["Country of asylum (ISO)"].isin(top_20_countries)
].copy()

top_20_totals_by_unit = (
    TotalAppsTop20
    .groupby("Cases / Persons")["applied"]
    .sum()
    .reindex(["P", "C"], fill_value=0)
)

TotalAppsTop20Persons = top_20_totals_by_unit["P"]
TotalAppsTop20Cases = top_20_totals_by_unit["C"]
TotalAppsTop20Mixed = (
    TotalAppsTop20Persons
    + TotalAppsTop20Cases
)

# All countries outside the top 20, also kept by reporting unit.
OtherCountriesPersons = TotalAppsPersons - TotalAppsTop20Persons
OtherCountriesCases = TotalAppsCases - TotalAppsTop20Cases
OtherCountriesMixed = OtherCountriesPersons + OtherCountriesCases

# Composition information for chart labels/tables.
worldwide_case_share = TotalAppsCases / TotalAppsMixed
top_20_case_share = TotalAppsTop20Cases / TotalAppsTop20Mixed
other_countries_case_share = OtherCountriesCases / OtherCountriesMixed

Worldwide 2000 Onwards - Plot the Top 20 Receiving Countries' Share of Asylum Applications as a Donut Chart

In [ ]:
# %% Annual Person and Case Applications and Decisions since 2000. Includes Pending Applicants from 1999 and
# earlier.
# Left axis = Persons; right axis = Cases.

apps_detail = (
    UNapps.reset_index()
    if isinstance(UNapps.index, pd.MultiIndex)
    else UNapps.copy()
)

decs_detail = (
    UNdecs.reset_index()
    if isinstance(UNdecs.index, pd.MultiIndex)
    else UNdecs.copy()
)

years = np.arange(1999, latest_available_year + 1)
positions = np.arange(len(years))
bar_width = 0.22

# Keep the three standard UNHCR application types, retaining all other codes
# in a separate category so applications are not omitted.
apps_detail["Application type group"] = np.select(
    [
        apps_detail["Application type"].eq("N"),
        apps_detail["Application type"].eq("R"),
        apps_detail["Application type"].eq("A"),
    ],
    [
        "New",
        "Repeat",
        "Appeal",
    ],
    default="Other / unspecified",
)

annual_applications = (
    apps_detail.loc[
        apps_detail["Year"].between(2000, latest_available_year)
    ]
    .groupby(
        ["Year", "Cases / Persons", "Application type group"],
        as_index=False,
    )["applied"]
    .sum()
)

annual_decisions = (
    decs_detail.loc[
        decs_detail["Year"].between(2000, latest_available_year)
    ]
    .groupby(["Year", "Cases / Persons"], as_index=False)["Total decisions"]
    .sum()
)

application_types = [
    "New",
    "Repeat",
    "Appeal",
    "Other / unspecified",
]

application_colours = {
    "New": "#D9E2F3",
    "Repeat": "#9DC3E6",
    "Appeal": "#4F81BD",
    "Other / unspecified": "#002664",
}

def application_series(unit_code, application_type):
    return (
        annual_applications.loc[
            annual_applications["Cases / Persons"].eq(unit_code)
            & annual_applications["Application type group"].eq(
                application_type
            )
        ]
        .groupby("Year")["applied"]
        .sum()
        .reindex(years, fill_value=0)
    )

def decision_series(unit_code):
    return (
        annual_decisions.loc[
            annual_decisions["Cases / Persons"].eq(unit_code)
        ]
        .groupby("Year")["Total decisions"]
        .sum()
        .reindex(years, fill_value=0)
    )

fig, ax_persons = plt.subplots(figsize=(20, 10))
ax_cases = ax_persons.twinx()

# Positions for each bar group.
person_application_x = positions - 0.345
person_decision_x = positions - 0.115
case_application_x = positions + 0.115
case_decision_x = positions + 0.345

# Persons: stacked applications by type, plus total decisions.
person_bottom = np.zeros(len(years))

for application_type in application_types:
    values = application_series("P", application_type)

    ax_persons.bar(
        person_application_x,
        values,
        width=bar_width,
        bottom=person_bottom,
        color=application_colours[application_type],
        label=application_type,
    )

    person_bottom += values.to_numpy()

person_decisions = decision_series("P")

# Pending asylum-seekers at the end of 1999 form the opening volume for 2000.
pending_1999 = pd.Series(0.0, index=years)
pending_1999.loc[1999] = opening_pending_2000

ax_persons.bar(
    positions,
    pending_1999,
    width=bar_width,
    color="#7030A0",
    label="Pending asylum-seekers — 1999",
)

ax_persons.bar(
    person_decision_x,
    person_decisions,
    width=bar_width,
    color="#00B050",
    label="Total Decisions — Persons",
)

# Cases: stacked applications by type, plus total decisions.
case_bottom = np.zeros(len(years))

for application_type in application_types:
    values = application_series("C", application_type)

    ax_cases.bar(
        case_application_x,
        values,
        width=bar_width,
        bottom=case_bottom,
        color=application_colours[application_type],
        hatch="//",
    )

    case_bottom += values.to_numpy()

case_decisions = decision_series("C")

ax_cases.bar(
    case_decision_x,
    case_decisions,
    width=bar_width,
    color="#70AD47",
    hatch="//",
    label="Total Decisions — Cases",
)

# Axis formatting.
ax_persons.set_ylabel("Reported Persons", color="#002664", fontsize=11)
ax_cases.set_ylabel("Reported Cases", color="#4F81BD", fontsize=11)

# Use one common scale for both reporting-unit axes.
common_ymax = max(
    person_bottom.max(),
    person_decisions.max(),
    case_bottom.max(),
    case_decisions.max(),
)
common_ymax *= 1.05  # modest headroom above the tallest bar

ax_persons.set_ylim(0, common_ymax)
ax_cases.set_ylim(0, common_ymax)

engineering_formatter = EngFormatter(unit="", places=1, sep="")

ax_persons.yaxis.set_major_formatter(engineering_formatter)
ax_cases.yaxis.set_major_formatter(engineering_formatter)

ax_persons.tick_params(axis="y", colors="#002664")
ax_cases.tick_params(axis="y", colors="#4F81BD")

year_step = 2 if len(years) > 15 else 1
display_years = np.r_[1999, years[(years >= 2000) & ((years - 2000) % year_step == 0)]]
display_positions = display_years - years[0]
ax_persons.set_xticks(display_positions)
ax_persons.set_xticklabels(display_years, rotation=0, ha="center")
ax_persons.set_xlabel("Calendar Year")

ax_persons.grid(axis="y", linestyle="--", alpha=0.35)
ax_persons.set_axisbelow(True)

ax_persons.spines["top"].set_visible(False)
ax_cases.spines["top"].set_visible(False)

# Legends.
application_type_handles = [
    mpl.patches.Patch(
        color=application_colours[application_type],
        label=f"Applications — {application_type}",
    )
    for application_type in application_types
] + [
    mpl.patches.Patch(
        color="#7030A0",
        label="Pending Applicants from 1999 and earlier",
    ),
    mpl.patches.Patch(
        facecolor="white",
        edgecolor="#1F1F1F",
        hatch="//",
        linewidth=0.8,
        label="Applications — Cases (hatched)",
    )
]

reporting_unit_handles = [
    mpl.patches.Patch(
        color="#00B050",
        label="Total Decisions — Persons",
    ),
    mpl.patches.Patch(
        facecolor="#70AD47",
        edgecolor="#1F1F1F",
        hatch="//",
        linewidth=0.8,
        label="Total Decisions — Cases",
    ),
]

application_legend = ax_persons.legend(
    handles=application_type_handles,
    title="Applications and Pending",
    loc="upper left",
    frameon=False,
    fontsize=9,
)
ax_persons.add_artist(application_legend)

ax_persons.legend(
    handles=reporting_unit_handles,
    title="Decisions",
    loc="upper left",
    bbox_to_anchor=(0.0, 0.79),
    ncols=1,
    frameon=False,
    fontsize=9,

)

fig.suptitle(
    f"Annual Applications by Type and Decisions, "
    f"1999–{latest_available_year}",
    fontsize=16,
    y=0.97,
)

ax_persons.set_title(
    "Applications are stacked as New, Repeat, Appeal, and Other/unspecified. "
    "The 1999 pending volume is persons only; cases use the right axis.",
    fontsize=10,
    pad=15,
    y=0.955,
)

fig.tight_layout()
plt.show()

In [ ]:
# %% Estimated-Person Applications by Type and Total Decisions, Annual since 2000 plus Pending Applicants from 1999 and earlier
# Case-reported values are converted using documented multipliers where
# available, with the fallback average multiplier for other case rows.

apps_estimated = (
    UNappsEstimated.reset_index()
    if isinstance(UNappsEstimated.index, pd.MultiIndex)
    else UNappsEstimated.copy()
)

decs_estimated = (
    UNdecsEstimated.reset_index()
    if isinstance(UNdecsEstimated.index, pd.MultiIndex)
    else UNdecsEstimated.copy()
)

years = np.arange(1999, latest_available_year + 1)
positions = np.arange(len(years))
bar_width = 0.38

# Retain N, R, and A explicitly. Other codes remain visible so that the
# stacked application total is complete.
apps_estimated["Application type group"] = np.select(
    [
        apps_estimated["Application type"].eq("N"),
        apps_estimated["Application type"].eq("R"),
        apps_estimated["Application type"].eq("A"),
    ],
    [
        "New",
        "Repeat",
        "Appeal",
    ],
    default="Other / unspecified",
)

application_types = [
    "New",
    "Repeat",
    "Appeal",
    "Other / unspecified",
]

application_colours = {
    "New": "#D9E2F3",
    "Repeat": "#9DC3E6",
    "Appeal": "#4F81BD",
    "Other / unspecified": "#002664",
}

annual_estimated_applications = (
    apps_estimated.loc[
        apps_estimated["Year"].between(2000, latest_available_year)
    ]
    .groupby(["Year", "Application type group"], as_index=False)[
        "applied (estimated persons)"
    ]
    .sum()
)

annual_estimated_decisions = (
    decs_estimated.loc[
        decs_estimated["Year"].between(2000, latest_available_year)
    ]
    .groupby("Year", as_index=False)["Total decisions (estimated persons)"]
    .sum()
)

def estimated_application_series(application_type):
    return (
        annual_estimated_applications.loc[
            annual_estimated_applications["Application type group"].eq(
                application_type
            )
        ]
        .groupby("Year")["applied (estimated persons)"]
        .sum()
        .reindex(years, fill_value=0)
    )

estimated_decisions = (
    annual_estimated_decisions
    .set_index("Year")["Total decisions (estimated persons)"]
    .reindex(years, fill_value=0)
)

fig, ax = plt.subplots(figsize=(18, 9))

# Stacked application bar.
application_bottom = np.zeros(len(years))

for application_type in application_types:
    application_values = estimated_application_series(application_type)

    ax.bar(
        positions - bar_width / 2,
        application_values,
        width=bar_width,
        bottom=application_bottom,
        color=application_colours[application_type],
        label=f"Applications — {application_type}",
    )

    application_bottom += application_values.to_numpy()

# Pending asylum-seekers at the end of 1999 form the opening volume for 2000.
pending_1999 = pd.Series(0.0, index=years)
pending_1999.loc[1999] = opening_pending_2000

ax.bar(
    positions,
    pending_1999,
    width=bar_width,
    color="#7030A0",
    label="Pending Applicants from 1999 and earlier",
)

# Total decisions bar.
ax.bar(
    positions + bar_width / 2,
    estimated_decisions,
    width=bar_width,
    color="#00B050",
    label="Total Decisions",
)

ax.set_ylabel("Estimated Persons")
ax.yaxis.set_major_formatter(
    EngFormatter(unit="", places=0, sep=" ")
)

year_step = 2 if len(years) > 15 else 1
display_years = np.r_[1999, years[(years >= 2000) & ((years - 2000) % year_step == 0)]]
display_positions = display_years - years[0]
ax.set_xticks(display_positions)
ax.set_xticklabels(display_years, rotation=0, ha="center")
ax.set_xlabel("Calendar Year")

ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.set_axisbelow(True)

for spine in ["top", "right"]:
    ax.spines[spine].set_visible(False)

ax.set_title(
    f"Estimated Persons by Application Type and Decisions, "
    f"1999–{latest_available_year}",
    fontsize=15,
    pad=18,
)

ax.text(
    0.5,
    0.99,
    (
        "Case-reported values converted using documented country/year "
        f"multipliers where available; otherwise {FALLBACK_CASE_TO_PERSON_MULTIPLIER:.2f} "
        "persons per case. The 1999 pending volume is reported persons."
    ),
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=9,
    color="#555555",
)

ax.legend(
    loc="upper left",
    ncols=1,
    frameon=False,
    fontsize=9,
)

fig.tight_layout()
plt.show()

In [ ]:
# %% Worldwide Top 20 Countries since 2000 - Share of Applications compared to All Other Countries

countries = [
    "Top 20 Countries",
    "All Other Countries",
]

data = [
    TotalAppsTop20Mixed,
    OtherCountriesMixed,
]

colours = [
    "#002664",
    "#7A7A7A",
]

fig, ax = plt.subplots(figsize=(9, 8), facecolor="white")
ax.set_facecolor("white")

wedges, _, percentages = ax.pie(
    data,
    startangle=165,
    colors=colours,
    autopct="%1.1f%%",
    pctdistance=0.72,
    wedgeprops={
        "edgecolor": "white",
        "linewidth": 2,
    },
    textprops={
        "color": "white",
        "fontsize": 14,
        "fontweight": "bold",
    },
)

ax.set_title(
    f"Share of Applications (Persons and Cases)\n"
    f"Top 20 Countries and All Other Countries, "
    f"2000–{latest_available_year}",
    fontsize=14,
    pad=22,
)


legend = ax.legend(
    wedges,
    [
        f"Top 20 Countries: {TotalAppsTop20Mixed:,.0f}",
        f"All Other Countries: {OtherCountriesMixed:,.0f}",
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, 0.98),
    ncols=2,
    frameon=True,
    fontsize=9,
    labelcolor="#000000",
)

legend.get_frame().set_facecolor("white")
legend.get_frame().set_edgecolor("white")
legend.get_frame().set_alpha(1)

plt.setp(legend.get_texts(), color="#000000")
fig.tight_layout()
plt.show()

Worldwide 2000 Onwards - Each year's Top 20 Application Receiving Countries

Worldwide 2000 Onwards - Create Datasets for each Year

In [ ]:
# Calculate each year's Top 20 receiving countries and their application share.
yearly_country_applications = (
    UNTotalAppsDecs.loc[
        UNTotalAppsDecs['Year'].between(YEAR_FROM, latest_available_year)
    ]
    .groupby(['Year', 'Country of asylum (ISO)'], as_index=False)['applied']
    .sum()
)

yearly_top_20 = (
    yearly_country_applications
    .sort_values(['Year', 'applied'], ascending=[True, False])
    .groupby('Year', group_keys=False)
    .head(20)
)

yearly_totals = (
    yearly_country_applications
    .groupby('Year', as_index=False)['applied']
    .sum()
    .rename(columns={'applied': 'Total applications'})
)

yearly_top_20_totals = (
    yearly_top_20
    .groupby('Year', as_index=False)['applied']
    .sum()
    .rename(columns={'applied': 'Top 20 applications'})
)

yearly_pie_data = (
    yearly_totals
    .merge(yearly_top_20_totals, on='Year', how='left')
    .fillna({'Top 20 applications': 0})
)

yearly_pie_data['Other countries applications'] = (
    yearly_pie_data['Total applications']
    - yearly_pie_data['Top 20 applications']
)

Worldwide 2000 Onwards - Show Share of Asylum Applications by Top 20 Receiving Countries each year compared with all
other Receiving Countries on a Faceted Pie Chart

In [ ]:
# Create one pie-chart facet for each year.
facet_columns = 4
facet_rows = int(np.ceil(len(yearly_pie_data) / facet_columns))

fig, axes = plt.subplots(
    facet_rows,
    facet_columns,
    figsize=(facet_columns * 4, facet_rows * 3.8),
)
facet_axes = np.atleast_1d(axes).ravel()

for ax, (_, year_data) in zip(facet_axes, yearly_pie_data.iterrows()):
    values = [
        year_data['Top 20 applications'],
        year_data['Other countries applications'],
    ]

    ax.pie(
        values,
        autopct=lambda pct: f'{pct:.0f}%',
        startangle=165,
        colors=('#002664', '#7A7A7A'),
        textprops={'fontsize': 9, 'color': 'white'},
        wedgeprops={'edgecolor': 'white', 'linewidth': 1},
    )
    ax.set_title(str(int(year_data['Year'])), fontsize=12)

for ax in facet_axes[len(yearly_pie_data):]:
    ax.set_visible(False)

fig.legend(
    handles=[
        mpl.patches.Patch(color='#002664', label='Top 20 countries'),
        mpl.patches.Patch(color='#7A7A7A', label='All Other Countries'),
    ],
    loc='upper center',
    bbox_to_anchor=(0.5, 0.97),
    ncols=2,
    frameon=False,
)

fig.suptitle(
    'Share of Applications: Each Year’s Top 20 Receiving Countries',
    fontsize=18,
    y=0.995,
)
fig.tight_layout(rect=(0, 0, 1, 0.95)) # Left, bottom, Right, top
plt.show()

Worldwide 2000 Onwards - Plot Worldwide Applications and Decision Volumes on a Horizontal Bar Chart

In [ ]:
# %% Worldwide Analysis 2000 to Last Calendar Year - Prepare Chart Datasets
# Worldwide analysis: create annual totals and outcome rates
worldwide_annual_data = (
    UNTotalAppsDecs.loc[
        UNTotalAppsDecs['Year'].between(2000, last_calendar_year)]
    .groupby('Year', as_index=False)[
        [
            'applied',
            'Total decisions',
            'Recognized decisions',
            'Rejected decisions',
            'Complementary protection',
            'Otherwise closed',
        ]
    ]
    .sum()
)
# Annual shortfall is the difference between new applications and decisions
# completed.
worldwide_annual_data['Shortfall'] = (
        worldwide_annual_data['applied'] - worldwide_annual_data[
    'Total decisions']
)
# Grant and refusal rates are calculated from worldwide decision totals each
# year.
worldwide_annual_data['Grant rate'] = (
    worldwide_annual_data['Recognized decisions']
    .div(worldwide_annual_data['Total decisions'])
    .where(worldwide_annual_data['Total decisions'].ne(0))
)
worldwide_annual_data['Refusal rate'] = (
    worldwide_annual_data['Rejected decisions']
    .div(worldwide_annual_data['Total decisions'])
    .where(worldwide_annual_data['Total decisions'].ne(0))
)
worldwide_annual_data['Other protection rate'] = (
    worldwide_annual_data['Complementary protection']
    .div(worldwide_annual_data['Total decisions'])
    .where(worldwide_annual_data['Total decisions'].ne(0))
)
worldwide_annual_data['Otherwise closed rate'] = (
    worldwide_annual_data['Otherwise closed']
    .div(worldwide_annual_data['Total decisions'])
    .where(worldwide_annual_data['Total decisions'].ne(0))
)
# Face-value productivity: decisions completed for each application received.
worldwide_annual_data['FVP'] = (
    worldwide_annual_data['Total decisions']
    .div(worldwide_annual_data['applied'])
    .where(worldwide_annual_data['applied'].ne(0))
)
worldwide_annual_data['Grant rate %'] = worldwide_annual_data[
    'Grant rate'].map(
    lambda rate: f'{rate:.0%}' if pd.notna(rate) else ''
)
worldwide_annual_data['Refusal rate %'] = worldwide_annual_data[
    'Refusal rate'].map(
    lambda rate: f'{rate:.0%}' if pd.notna(rate) else ''
)
worldwide_annual_data['Other protection rate %'] = worldwide_annual_data[
    'Other protection rate'
].map(lambda rate: f'{rate:.0%}' if pd.notna(rate) else '')
worldwide_annual_data['Otherwise closed rate %'] = worldwide_annual_data[
    'Otherwise closed rate'
].map(lambda rate: f'{rate:.0%}' if pd.notna(rate) else '')
worldwide_annual_data['FVP %'] = worldwide_annual_data['FVP'].map(
    lambda value: f'{value:.0%}' if pd.notna(value) else ''
)

In [ ]:
# %% Worldwide 2000 onwards — Applications by Type and Decisions

apps_for_worldwide_chart = (
    UNapps.reset_index()
    if isinstance(UNapps.index, pd.MultiIndex)
    else UNapps.copy()
)

apps_for_worldwide_chart["Application type group"] = np.select(
    [
        apps_for_worldwide_chart["Application type"].eq("N"),
        apps_for_worldwide_chart["Application type"].eq("R"),
        apps_for_worldwide_chart["Application type"].eq("A"),
    ],
    [
        "New",
        "Repeat",
        "Appeal",
    ],
    default="Other / unspecified",
)

worldwide_applications_by_type = (
    apps_for_worldwide_chart.loc[
        apps_for_worldwide_chart["Year"].between(
            2000,
            last_calendar_year,
        )
    ]
    .groupby(
        ["Year", "Application type group"],
        as_index=False,
    )["applied"]
    .sum()
)

chart_years = worldwide_annual_data["Year"].to_numpy()
annual_bar_positions = np.arange(len(chart_years))
annual_bar_width = 0.42

application_types = [
    "New",
    "Repeat",
    "Appeal",
    "Other / unspecified",
]

application_colours = {
    "New": "#D9E2F3",
    "Repeat": "#9DC3E6",
    "Appeal": "#4F81BD",
    "Other / unspecified": "#002664",
}

fig, ax = plt.subplots(figsize=(18, 9), facecolor="white")
ax.set_facecolor("white")

application_bottom = np.zeros(len(chart_years))

for application_type in application_types:
    application_values = (
        worldwide_applications_by_type.loc[
            worldwide_applications_by_type[
                "Application type group"
            ].eq(application_type)
        ]
        .set_index("Year")["applied"]
        .reindex(chart_years, fill_value=0)
    )

    ax.bar(
        annual_bar_positions - annual_bar_width / 2,
        application_values,
        width=annual_bar_width,
        bottom=application_bottom,
        color=application_colours[application_type],
        label=f"Applications — {application_type}",
    )

    application_bottom += application_values.to_numpy()

ax.bar(
    annual_bar_positions + annual_bar_width / 2,
    worldwide_annual_data["Total decisions"],
    width=annual_bar_width,
    color="#732282",
    label="Total Decisions",
)

ax.set_xticks(annual_bar_positions)
ax.set_xticklabels(chart_years, rotation=0)
ax.yaxis.set_major_formatter(EngFormatter(unit="", places=0, sep=" "))

ax.set_xlabel("Calendar Year", fontsize=12)
ax.set_ylabel("Reported Applications and Decisions", fontsize=12)

ax.set_title(
    f"Worldwide Applications by Type and Decisions, "
    f"2000–{last_calendar_year}",
    fontsize=16,
    pad=16,
)

ax.text(
    0.5,
    0.985,
    "Applications include persons and cases.",
    transform=ax.transAxes,
    ha="center",
    va="bottom",
    fontsize=9,
    color="#555555",
)

ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.set_axisbelow(True)

ax.legend(
    loc="upper center",
    bbox_to_anchor=(0.5, 0.98),
    ncols=5,
    frameon=False,
    fontsize=9,
)

fig.tight_layout()

for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)

fig.tight_layout()
plt.show()

Worldwide 2000 Onwards - Calculate Grant Rates, Rates of Other Protection, Refusal and Otherwise Closed Rates

In [ ]:
# %% Plot combined worldwide decision-outcome rates
worldwide_total_decisions = worldwide_annual_data['Total decisions'].sum()
worldwide_total_granted = worldwide_annual_data['Recognized decisions'].sum()
worldwide_total_other_protection = worldwide_annual_data[
    'Complementary protection'].sum()
worldwide_total_refused = worldwide_annual_data['Rejected decisions'].sum()
worldwide_total_otherwise_closed = worldwide_annual_data[
    'Otherwise closed'].sum()

worldwide_outcome_rates = [
    worldwide_total_granted / worldwide_total_decisions,
    worldwide_total_other_protection / worldwide_total_decisions,
    worldwide_total_refused / worldwide_total_decisions,
    worldwide_total_otherwise_closed / worldwide_total_decisions,
]
worldwide_outcome_labels = [
    'Granted',
    'Other Protection',
    'Refused',
    'Closed',
]
worldwide_outcome_colors = ['#00B050', '#FFD700', '#C53A3A', '#7A7A7A']

Worldwide 2000 Onwards - Plot Worldwide Combined Decision Outcome Rates (Grants, Other Protection, Refusals,
Otherwise Closed) on a single Pie Chart

In [ ]:
# %% Combined Decision Outcome Rates 2000 to latest calendar year, Pie Chart.
fig, ax = plt.subplots(figsize=(8, 8))
wedges, _, percentage_labels = ax.pie(
    worldwide_outcome_rates,
    labels=worldwide_outcome_labels,
    autopct='%1.1f%%',
    startangle=90,
    counterclock=False,
    colors=worldwide_outcome_colors,
    wedgeprops={'edgecolor': 'white', 'linewidth': 2},
    textprops={'fontsize': 8},
)
plt.setp(percentage_labels, color='white', fontweight='bold')
ax.set_title(
    f'Worldwide Decision Outcomes, 2000–{last_calendar_year}',
    fontsize=12,
    pad=10,
)
facet_legend_handles = [
    mpl.patches.Patch(color=color, label=label)
    for label, color in zip(worldwide_outcome_labels,
                            worldwide_outcome_colors)]
fig.legend(
    handles=facet_legend_handles,
    loc = 'upper center',
	bbox_to_anchor=(0.6, 0.84),
    ncols=4,
    frameon=False,
	fontsize=8
)
fig.tight_layout(rect = (0.2, 0.2, 0.98, 0.90)) # Left, bottom, right, top
print('-' * 100)
plt.show()
print('-' * 100)

Worldwide 2000 Onwards - Prepare Annual Decision Outcome Rates (Grants, Other Protection, Refusals,
Otherwise Closed)

Worldwide 2000 Onwards - Plot Annual Decision Outcome (Grants, Other Protection, Refusals, Otherwised Closed) Volumes
 on a Combination Chart (Vertical Bar and Line)

In [ ]:
# %% Decision Outcomes by Year, 2000 to latest calendar year, faceted Pie Chart
facet_columns = 4
facet_rows = int(np.ceil(len(worldwide_annual_data) / facet_columns))
fig, axes = plt.subplots(
    facet_rows,
    facet_columns,
    figsize=(facet_columns * 4, facet_rows * 3.8),
)
facet_axes = np.atleast_1d(axes).ravel()

for ax, (_, annual_data) in zip(facet_axes, worldwide_annual_data.iterrows()):
    annual_outcome_rates = [
        annual_data['Recognized decisions'] / annual_data['Total decisions'],
        annual_data['Complementary protection'] / annual_data[
            'Total decisions'],
        annual_data['Rejected decisions'] / annual_data['Total decisions'],
        annual_data['Otherwise closed'] / annual_data['Total decisions'],
    ]
    ax.pie(
        annual_outcome_rates,
        autopct=lambda
            percentage: f'{percentage:.0f}%' if percentage >= 4 else '',
        startangle=90,
        counterclock=False,
        colors=worldwide_outcome_colors,
        wedgeprops={'edgecolor': 'white', 'linewidth': 1},
        textprops={'fontsize': 12},
    )
    ax.set_title(str(int(annual_data['Year'])), fontsize=14, pad=6)

for ax in facet_axes[len(worldwide_annual_data):]:
    ax.set_visible(False)

facet_legend_handles = [
    mpl.patches.Patch(color=color, label=label)
    for label, color in zip(worldwide_outcome_labels,
                            worldwide_outcome_colors)
]
fig.legend(
    handles=facet_legend_handles,
    loc = 'upper center',
	bbox_to_anchor=(0.5, 0.97),
    ncols=4,
    frameon=False,
)
fig.suptitle(
    f'Worldwide Decision Outcomes by Year, 2000–{last_calendar_year}',
    fontsize=16,
    y=0.995,
)
fig.tight_layout(rect=(0, 0.05, 1, 0.95))
plt.show()

In [ ]:
# %% Plot worldwide annual decision-outcome volumes (not percentage stacked)
worldwide_outcome_columns = [
    'Recognized decisions',
    'Complementary protection',
    'Rejected decisions',
    'Otherwise closed',
]
worldwide_outcome_labels = [
    'Granted',
    'Other Protection',
    'Refused',
    'Closed',
]
# Set colours for the bars
worldwide_outcome_colors = ['#00B050', '#FFD700', '#C53A3A', '#7A7A7A']
fig, ax = plt.subplots(figsize=(16, 9))
bottom = np.zeros(len(worldwide_annual_data))
for column, label, color in zip(
        worldwide_outcome_columns,
        worldwide_outcome_labels,
        worldwide_outcome_colors,
):
    values = worldwide_annual_data[column]
    bars = ax.bar(
        worldwide_annual_data['Year'],
        values,
        bottom=bottom,
        label=label,
        color=color,
        width=0.8,
    )
    ax.bar_label(
        bars,
        labels=[
            format_volume_label(value)
            if value >= total_decisions * 0.04 else ''
            for value, total_decisions in zip(
                values,
                worldwide_annual_data['Total decisions'],
            )
        ],
        label_type='center',
        fontsize=8,
        color='#000000' if column in ('Recognized decisions',
                                      'Complementary protection') else
        'white',
    )
    bottom += values.to_numpy()

applications_axis = ax.twinx()
applications_line, = applications_axis.plot(
    worldwide_annual_data['Year'],
    worldwide_annual_data['applied'],
    color='#002664',
    linewidth=2.5,
    linestyle=':',
    marker='o',
    markersize=4,
    label='Applications',
)
applications_axis.yaxis.set_major_formatter(EngFormatter(unit = '', places = 0))
applications_axis.set_ylabel('Applications', fontsize=12)
applications_axis.grid(False)
applications_axis.spines['top'].set_visible(False)

# Both axes use the same zero-based scale for direct visual comparison.
common_ymax = max(
    bottom.max(),
    worldwide_annual_data['applied'].max(),
) * 1.05
ax.set_ylim(0, common_ymax)
applications_axis.set_ylim(0, common_ymax)

ax.yaxis.set_major_formatter(EngFormatter(unit = '', places = 0))
ax.set_xlabel('Calendar Year', fontsize=12)
ax.set_ylabel('Decisions', fontsize=12)
ax.set_title(
    f'Worldwide Decisions by Applications and Outcome, 2000–'
    f'{last_calendar_year}',
    fontsize=14,
    pad=16,
)
decision_handles, decision_labels = ax.get_legend_handles_labels()
ax.legend(
    decision_handles + [applications_line],
    decision_labels + ['Applications'],
    loc='upper left',
    ncols=3,
    frameon=False,
)
ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.set_axisbelow(True)
ax.tick_params(axis='x', rotation=0)
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
fig.tight_layout()
print('-' * 100)
plt.show()
print('-' * 100)

Worldwide 2000 Onwards - Plot Worldwide Decision Outcome Rates Faceted by Year

Worldwide 2000 Onwards - Plot Annual Decision Outcome Rates as a Stacked Bar Chart

In [ ]:
# %% Plot worldwide annual decision-outcome rates as a stacked bar chart
worldwide_outcome_rate_columns = [
    'Grant rate',
    'Other protection rate',
    'Refusal rate',
    'Otherwise closed rate',
]
# Set colours for the bars
worldwide_outcome_rate_colors = ['#00B050', '#FFD700', '#C53A3A', '#7A7A7A']
fig, ax = plt.subplots(figsize=(16, 9))
rate_bottom = np.zeros(len(worldwide_annual_data))
for column, label, color in zip(
        worldwide_outcome_rate_columns,
        worldwide_outcome_labels,
        worldwide_outcome_rate_colors,
):
    rates = worldwide_annual_data[column].fillna(0)
    bars = ax.bar(
        worldwide_annual_data['Year'],
        rates,
        bottom=rate_bottom,
        label=label,
        color=color,
        width=0.8,
    )
    ax.bar_label(
        bars,
        labels=[f'{rate:.0%}' if rate >= 0.04 else '' for rate in rates],
        label_type='center',
        fontsize=8,
        color='#000000' if column in ('Grant rate',
                                      'Other protection rate') else 'white',
    )
    rate_bottom += rates.to_numpy()
# Set axes etc.
ax.set_ylim(0, 1)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel('Calendar Year', fontsize=12)
ax.set_ylabel('Share of Decisions', fontsize=12)
ax.set_title(
    f'Worldwide Decision Outcome Rates by Year, 20'
    f'00–{last_calendar_year}',
    fontsize=16,
    pad=48,
)
ax.set_xticks(worldwide_annual_data['Year'])
ax.tick_params(axis='x', rotation=0)
ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.set_axisbelow(True)
ax.legend(
    loc='upper center',
    bbox_to_anchor=(0.5, 1.08),
    ncols=4,
    frameon=False,
    fontsize=10,
    title_fontsize=11,
)
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
fig.tight_layout()
print('-' * 100)
plt.show()
print('-' * 100)

Worldwide 2000 Onwards - Plot Face-Value Productivity Rates (FVP) as a Smoothed Line Chart

In [ ]:
# %% Worldwide 2000 Onwards - Plot Worldwide Face-Value Productivity Rates
# FVP compares decisions with applications recorded in the same calendar year.
worldwide_fvp = (
    worldwide_annual_data["Total decisions"]
    .div(worldwide_annual_data["applied"])
    .where(worldwide_annual_data["applied"].ne(0))
)
average_worldwide_fvp = worldwide_fvp.mean()

worldwide_fvp_smoothed = worldwide_fvp.rolling(
    window=3,
    center=True,
    min_periods=1,
).mean()

fig, ax = plt.subplots(figsize=(16, 9))
ax.plot(
    worldwide_annual_data['Year'],
    worldwide_fvp_smoothed,
    linewidth=2.5,
    color='#002664',
    label='Worldwide FVP (3-year moving average)',
)
ax.axhline(
    y=1,
    color='#882345',
    linestyle='--',
    linewidth=1.5,
    label='100% FVP',
)
ax.axhline(
    y=average_worldwide_fvp,
    color='#E67E22',
    linestyle='--',
    linewidth=1.5,
    label=f'Average annual FVP ({average_worldwide_fvp:.1%})',
)
ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
ax.set_xlabel('Calendar Year', fontsize=12, rotation=0)
ax.set_ylabel('Face-value Productivity', fontsize=12)
ax.set_title(
    f'Worldwide Face-Value Productivity, '
    f'2000–{last_calendar_year}',
    fontsize=12,
    pad=16,
)

ax.grid(axis='y', linestyle='--', alpha=0.35)
ax.set_axisbelow(True)
ax.legend(frameon=False)
ax.tick_params(axis='x', rotation=0)
for spine in ('top', 'right'):
    ax.spines[spine].set_visible(False)
fig.tight_layout()
plt.show()

Worldwide 2000 Onwards - Plot Annual Applications minus Decisions as a Bar Chart

In [ ]:
# %% Worldwide 2000 onwards - Plot Worldwide Annual Application Shortfall
shortfall_colors = np.where(
    worldwide_annual_data["Shortfall"].ge(0),
    "#C53A3A",
    "#00B050",
)

fig, ax = plt.subplots(figsize=(16, 9))
ax.bar(
    worldwide_annual_data["Year"],
    worldwide_annual_data["Shortfall"],
    color=shortfall_colors,
    width=0.8,
)
ax.axhline(0, color="#000000", linewidth=0.8)
ax.yaxis.set_major_formatter(EngFormatter(unit="", places=0))
ax.set_xlabel("Calendar Year", fontsize=12)
ax.set_ylabel("Applications minus decisions", fontsize=12)
ax.set_title(
    f"Worldwide Annual Applications minus Decisions, "
    f"2000–{last_calendar_year}",
    fontsize=16,
    pad=16,
)
ax.set_xticks(worldwide_annual_data["Year"])
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.set_axisbelow(True)
ax.legend(
    handles=[
        mpl.patches.Patch(
            color="#C53A3A",
            label="Applications exceed decisions",
        ),
        mpl.patches.Patch(
            color="#00B050",
            label="Decisions exceed applications",
        ),
    ],
    loc="upper left",
    frameon=False,
)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)

fig.tight_layout()
plt.show()

Worldwide 2000 onwards - Plot Worldwide Annual Applications minus Decisions as a Waterfall Chart

### Worldwide waterfall using observed UNHCR asylum-seeker stocks

This chart starts with the observed end-2000 UNHCR stock, displays annual applications-minus-decisions flows from 2001, and shows the observed annual UNHCR stock line. It deliberately has no modelled Total column.

In [ ]:
# %% Worldwide annual application-flow waterfall with observed UNHCR stocks
# UNHCR's country-of-asylum population records are aggregated to a worldwide
# end-year asylum-seeker stock.  The stock is observed; the annual bars remain
# applications minus decisions and are not a calculated backlog.

worldwide_unhcr_population_records = []
worldwide_stock_years = np.arange(2000, last_calendar_year + 1)

for stock_year in worldwide_stock_years:
    response = requests.get(
        f"{API_BASE_URL}/population/",
        params={
            "yearFrom": int(stock_year),
            "yearTo": int(stock_year),
            "coa_all": "true",
            "coo_all": "true",
            "limit": 10000,
        },
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    response.raise_for_status()
    response_json = response.json()
    worldwide_unhcr_population_records.extend(response_json["items"])

    for page_number in range(2, response_json["maxPages"] + 1):
        page_response = requests.get(
            f"{API_BASE_URL}/population/",
            params={
                "yearFrom": int(stock_year),
                "yearTo": int(stock_year),
                "coa_all": "true",
                "coo_all": "true",
                "limit": 10000,
                "page": page_number,
            },
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
        page_response.raise_for_status()
        worldwide_unhcr_population_records.extend(
            page_response.json()["items"]
        )

worldwide_unhcr_population = pd.DataFrame(
    worldwide_unhcr_population_records
)
worldwide_unhcr_population["asylum_seekers"] = pd.to_numeric(
    worldwide_unhcr_population["asylum_seekers"],
    errors="coerce",
).fillna(0)

worldwide_unhcr_stocks = (
    worldwide_unhcr_population
    .groupby("year", as_index=False)["asylum_seekers"]
    .sum()
    .rename(
        columns={
            "year": "Year",
            "asylum_seekers": "UNHCR reported asylum-seeker stock",
        }
    )
)
worldwide_unhcr_stocks.loc[
    worldwide_unhcr_stocks["UNHCR reported asylum-seeker stock"].eq(0),
    "UNHCR reported asylum-seeker stock",
] = np.nan

unhcr_end_2000_worldwide_stock = worldwide_unhcr_stocks.loc[
    worldwide_unhcr_stocks["Year"].eq(2000),
    "UNHCR reported asylum-seeker stock",
].iloc[0]

worldwide_flow_after_2000 = worldwide_annual_data.loc[
    worldwide_annual_data["Year"].gt(2000)
].copy()
worldwide_opening_balance = (
    unhcr_end_2000_worldwide_stock
    + worldwide_flow_after_2000["Shortfall"].cumsum().shift(fill_value=0)
)
worldwide_flow_colors = np.where(
    worldwide_flow_after_2000["Shortfall"].ge(0),
    "#C53A3A",
    "#00B050",
)

fig, ax = plt.subplots(figsize=(16, 9))

opening_bar = ax.bar(
    2000,
    unhcr_end_2000_worldwide_stock,
    color="#7030A0",
    width=0.8,
    zorder=3,
)
ax.bar_label(
    opening_bar,
    labels=[format_volume_label(unhcr_end_2000_worldwide_stock)],
    padding=3,
    fontsize=9,
)

ax.bar(
    worldwide_flow_after_2000["Year"],
    worldwide_flow_after_2000["Shortfall"],
    bottom=worldwide_opening_balance,
    color=worldwide_flow_colors,
    width=0.8,
)

ax.plot(
    worldwide_unhcr_stocks["Year"],
    worldwide_unhcr_stocks["UNHCR reported asylum-seeker stock"],
    color="#7030A0",
    linewidth=2.2,
    marker="o",
    markersize=3.5,
    zorder=5,
)

latest_worldwide_stock = worldwide_unhcr_stocks.dropna(
    subset=["UNHCR reported asylum-seeker stock"]
).iloc[-1]
ax.annotate(
    format_volume_label(
        latest_worldwide_stock["UNHCR reported asylum-seeker stock"]
    ),
    xy=(
        latest_worldwide_stock["Year"],
        latest_worldwide_stock["UNHCR reported asylum-seeker stock"],
    ),
    xytext=(5, 0),
    textcoords="offset points",
    ha="left",
    va="center",
    fontsize=9,
    color="#4A235A",
    fontweight="bold",
)

ax.axhline(0, color="#000000", linewidth=0.8)
ax.yaxis.set_major_formatter(EngFormatter(unit="", places=0))
ax.set_xlabel("Calendar Year", fontsize=12)
ax.set_ylabel("Observed stock + annual net flow", fontsize=12)
ax.set_title(
    f"Worldwide Annual Asylum Application Flow Waterfall with UNHCR "
    f"Stocks, 2000–{last_calendar_year}",
    fontsize=15,
    pad=16,
)
waterfall_year_ticks = np.unique(
    np.append(
        np.arange(2000, last_calendar_year + 1, 5),
        last_calendar_year,
    )
)
ax.set_xticks(waterfall_year_ticks)
ax.tick_params(axis="x", rotation=0)
ax.grid(axis="y", linestyle="--", alpha=0.35)
ax.set_axisbelow(True)
ax.legend(
    handles=[
        mpl.patches.Patch(
            color="#7030A0",
            label="Observed UNHCR end-2000 asylum-seeker stock",
        ),
        mpl.patches.Patch(
            color="#C53A3A",
            label="Applications exceed decisions",
        ),
        mpl.patches.Patch(
            color="#00B050",
            label="Decisions exceed applications",
        ),
        mpl.lines.Line2D(
            [], [],
            color="#7030A0",
            marker="o",
            linewidth=2.2,
            label="Observed UNHCR end-year asylum-seeker stock",
        ),
    ],
    loc="upper left",
    frameon=False,
)
for spine in ("top", "right"):
    ax.spines[spine].set_visible(False)

fig.text(
    0.5,
    0.01,
    (
        "Purple line: observed UNHCR end-year asylum-seeker stock. "
        "Bars: annual applications minus decisions from 2001 onward. "
        "No modelled total is shown."
    ),
    ha="center",
    fontsize=9,
    color="#4A235A",
)
fig.tight_layout(rect=(0.02, 0.04, 0.99, 0.98))
plt.show()

In [ ]:
# %% Calculate annual FVP for the top 20 countries worldwide
# FVP is total decisions divided by applications in the same calendar year.
# No historical pending stock is added to the denominator.

top_20_worldwide_codes = top_20_worldwide_countries[
    "Country of asylum (ISO)"
]

top_20_worldwide_fvp = (
    UNTotalAppsDecs.loc[
        UNTotalAppsDecs["Year"].between(2000, latest_available_year)
        & UNTotalAppsDecs["Country of asylum (ISO)"].isin(
            top_20_worldwide_codes
        )
    ]
    .groupby(["Year", "Country of asylum (ISO)"], as_index=False)[
        ["applied", "Total decisions"]
    ]
    .sum()
)

top_20_worldwide_fvp["FVP"] = (
    top_20_worldwide_fvp["Total decisions"]
    .div(top_20_worldwide_fvp["applied"])
    .where(top_20_worldwide_fvp["applied"].ne(0))
)

top_20_worldwide_fvp["FVP smoothed"] = (
    top_20_worldwide_fvp
    .groupby("Country of asylum (ISO)")["FVP"]
    .transform(
        lambda fvp: fvp.rolling(
            window=3,
            center=True,
            min_periods=1,
        ).mean()
    )
)

Individual Countries - Plot Annual Face-Value Productivity (FVP) for each Top 20 Receiving Country

In [ ]:
# %% Individual Countries - Plot Annual FVP for each Top 20 Receiving Country
facet_columns = 3
facet_rows = int(np.ceil(len(top_20_worldwide_countries) / facet_columns))
fig, axes = plt.subplots(
    facet_rows,
    facet_columns,
    figsize=(facet_columns * 7, facet_rows * 3.2),
    sharex=True,
)
facet_axes = np.atleast_1d(axes).ravel()

fvp_year_ticks = np.unique(
    np.append(
        np.arange(2000, latest_available_year + 1, 5),
        latest_available_year,
    )
)

for ax, country_code in zip(
        facet_axes,
        top_20_worldwide_countries["Country of asylum (ISO)"],
):
    country_fvp = top_20_worldwide_fvp.loc[
        top_20_worldwide_fvp["Country of asylum (ISO)"].eq(country_code)
    ]
    average_country_fvp = country_fvp["FVP"].mean()

    ax.plot(
        country_fvp["Year"],
        country_fvp["FVP smoothed"],
        marker="o",
        markersize=3,
        linewidth=1.8,
        color="#732282" if country_code == "GBR" else "#002664",
    )
    ax.axhline(y=1, color="#882345", linestyle="--", linewidth=1)
    ax.axhline(
        y=average_country_fvp,
        color="#E67E22",
        linestyle="--",
        linewidth=1,
    )
    ax.set_title(country_code, fontsize=10, pad=5)
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    ax.set_axisbelow(True)
    ax.set_xticks(fvp_year_ticks)
    ax.tick_params(axis="x", rotation=0, labelsize=8)
    ax.tick_params(axis="y", labelsize=8)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))

    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

for ax in facet_axes[len(top_20_worldwide_countries):]:
    ax.set_visible(False)

fig.supxlabel("Calendar Year", fontsize=10)
fig.supylabel("Face-value Productivity (FVP)", fontsize=10)
fig.suptitle(
    f"Face-Value Productivity (3-Year Moving Average): Top 20 Countries, "
    f"2000–{latest_available_year}",
    fontsize=12,
    y=0.995,
)
fig.text(
    0.5,
    0.972,
    "FVP = total decisions divided by applications in the same calendar year. "
    "Orange dashed: each country's average annual FVP.",
    ha="center",
    fontsize=8,
    color="#4A235A",
)
fig.tight_layout(rect=(0.01, 0.03, 0.99, 0.94))
fig.subplots_adjust(wspace=0.2, hspace=0.3)
plt.show()

Last Calendar Year - Analyse Most Recent Year

Last Calendar Year - Prepare Datasets

In [ ]:
# %% Individual Countries - Prepare annual shortfalls for Top 20 receiving countries
top_20_country_shortfall_data = (
    UNTotalAppsDecs.loc[
        UNTotalAppsDecs['Year'].between(2000, last_calendar_year)
        & UNTotalAppsDecs['Country of asylum (ISO)'].isin(
            top_20_worldwide_codes
        )
    ]
    .groupby(['Year', 'Country of asylum (ISO)'], as_index=False)[
        ['applied', 'Total decisions']
    ]
    .sum()
    .sort_values(['Country of asylum (ISO)', 'Year'])
)

top_20_country_shortfall_data['Shortfall'] = (
    top_20_country_shortfall_data['applied']
    - top_20_country_shortfall_data['Total decisions']
)

# The 1999 historical pending dataset reports one worldwide total only.
# Model country shares from each country's 2000 estimated-person applications.
estimated_2000_applications_by_country = (
    UNappsEstimated.loc[UNappsEstimated['Year'].eq(2000)]
    .groupby('Country of asylum (ISO)')['applied (estimated persons)']
    .sum()
)
estimated_1999_pending_by_country = (
    estimated_2000_applications_by_country
    .div(estimated_2000_applications_by_country.sum())
    .mul(opening_pending_2000)
)

top_20_country_shortfall_data['Estimated pending applicants, end-1999'] = (
    top_20_country_shortfall_data['Country of asylum (ISO)']
    .map(estimated_1999_pending_by_country)
    .fillna(0)
)

minimum_backlog_last_calendar_year = (
    top_20_country_shortfall_data
    .groupby('Country of asylum (ISO)', as_index=False)[
        ['applied', 'Total decisions']
    ]
    .sum()
    .rename(columns={
        'applied': 'Applications since 2000',
        'Total decisions': 'Decisions since 2000',
    })
)
minimum_backlog_last_calendar_year['Estimated opening pending, end-1999'] = (
    minimum_backlog_last_calendar_year['Country of asylum (ISO)']
    .map(estimated_1999_pending_by_country)
    .fillna(0)
)

minimum_backlog_last_calendar_year['Net shortfall since 2000'] = (
    minimum_backlog_last_calendar_year['Applications since 2000']
    - minimum_backlog_last_calendar_year['Decisions since 2000']
)
minimum_backlog_last_calendar_year['Modelled implied backlog at last calendar year'] = (
    minimum_backlog_last_calendar_year[
        'Estimated opening pending, end-1999'
    ]
    + minimum_backlog_last_calendar_year['Net shortfall since 2000']
)
minimum_backlog_last_calendar_year['Minimum backlog at last calendar year'] = (
    minimum_backlog_last_calendar_year[
        'Modelled implied backlog at last calendar year'
    ]
    .clip(lower=0)
)
minimum_backlog_last_calendar_year = (
    minimum_backlog_last_calendar_year.sort_values(
        'Minimum backlog at last calendar year',
        ascending=False,
    )
)

In [ ]:
# %% Load and cache UNHCR annual end-year asylum-seeker stocks
# UNHCR defines asylum-seekers as people whose claims for international
# protection have not yet been adjudicated. Country-level 1999 values are not
# usable in the API, so this observed series begins at end-2000.

UNHCR_STOCK_CACHE_FILE = (
    OBSERVED_DATA_CACHE_DIR / 'unhcr_asylum_seeker_stocks.parquet'
)
UNHCR_STOCK_METADATA_FILE = (
    OBSERVED_DATA_CACHE_DIR / 'unhcr_asylum_seeker_stocks_metadata.json'
)
unhcr_stock_release_cutoff = get_latest_release_date()

# Include Rwanda and Australia so their individual charts use the same
# observed UNHCR stock series as the Top 20 charts.
stock_country_codes = list(dict.fromkeys([
    *top_20_worldwide_codes,
    'RWA',
    'AUS',
]))

if cache_is_fresh(
        UNHCR_STOCK_CACHE_FILE,
        release_cutoff=unhcr_stock_release_cutoff,
):
    unhcr_annual_population = pd.read_parquet(UNHCR_STOCK_CACHE_FILE)
    unhcr_stock_cache_source = 'local cache'
    unhcr_stock_retrieved_at = datetime.datetime.fromtimestamp(
        UNHCR_STOCK_CACHE_FILE.stat().st_mtime,
        tz=datetime.timezone.utc,
    )
else:
    unhcr_countries_response = requests.get(
        f'{API_BASE_URL}/countries/',
        params={'limit': 400},
        timeout=REQUEST_TIMEOUT_SECONDS,
    )
    unhcr_countries_response.raise_for_status()
    unhcr_country_codes = pd.DataFrame(
        unhcr_countries_response.json()['items']
    )
    unhcr_iso_to_code = (
        unhcr_country_codes
        .dropna(subset=['iso', 'code'])
        .drop_duplicates('iso')
        .set_index('iso')['code']
    )

    unhcr_annual_population_records = []
    for country_code in stock_country_codes:
        unhcr_coa_code = unhcr_iso_to_code.get(country_code)
        if pd.isna(unhcr_coa_code):
            raise ValueError(
                f'UNHCR does not provide a country code for {country_code}.'
            )

        request_params = {
            'yearFrom': 2000,
            'yearTo': last_calendar_year,
            'coa': unhcr_coa_code,
            'coo_all': 'true',
            'limit': 10000,
        }
        response = requests.get(
            f'{API_BASE_URL}/population/',
            params=request_params,
            timeout=REQUEST_TIMEOUT_SECONDS,
        )
        response.raise_for_status()
        response_json = response.json()
        unhcr_annual_population_records.extend(response_json['items'])

        for page_number in range(2, response_json['maxPages'] + 1):
            request_params['page'] = page_number
            response = requests.get(
                f'{API_BASE_URL}/population/',
                params=request_params,
                timeout=REQUEST_TIMEOUT_SECONDS,
            )
            response.raise_for_status()
            unhcr_annual_population_records.extend(
                response.json()['items']
            )

    unhcr_annual_population = (
        pd.DataFrame(unhcr_annual_population_records)[
            ['year', 'coa_iso', 'asylum_seekers']
        ]
        .copy()
    )
    # The full API response mixes text and numeric population fields. Cache
    # only the three fields used here, after normalising the stock to numeric.
    unhcr_annual_population['asylum_seekers'] = pd.to_numeric(
        unhcr_annual_population['asylum_seekers'],
        errors='coerce',
    ).fillna(0)
    unhcr_stock_retrieved_at = datetime.datetime.now(datetime.timezone.utc)
    unhcr_annual_population.to_parquet(UNHCR_STOCK_CACHE_FILE, index=False)
    save_snapshot(
        unhcr_annual_population,
        'unhcr_asylum_seeker_stocks',
        unhcr_stock_retrieved_at,
    )
    save_refresh_metadata(
        UNHCR_STOCK_METADATA_FILE,
        {
            'dataset': 'UNHCR population API: asylum seekers',
            'api_url': f'{API_BASE_URL}/population/',
            'retrieved_at_utc': unhcr_stock_retrieved_at.isoformat(),
            'release_cutoff_utc': unhcr_stock_release_cutoff.isoformat(),
            'countries_requested': ','.join(stock_country_codes),
            'year_from': 2000,
            'year_to_requested': last_calendar_year,
            'row_count': len(unhcr_annual_population),
        },
    )
    unhcr_stock_cache_source = 'UNHCR API refresh'

required_unhcr_columns = {'year', 'coa_iso', 'asylum_seekers'}
missing_unhcr_columns = required_unhcr_columns.difference(
    unhcr_annual_population.columns
)
if missing_unhcr_columns:
    raise ValueError(
        'UNHCR population extract is missing required columns: '
        f'{sorted(missing_unhcr_columns)}'
    )

unhcr_annual_population['asylum_seekers'] = pd.to_numeric(
    unhcr_annual_population['asylum_seekers'],
    errors='coerce',
).fillna(0)

unhcr_annual_asylum_seeker_stocks = (
    unhcr_annual_population
    .groupby(['year', 'coa_iso'], as_index=False)['asylum_seekers']
    .sum()
    .rename(
        columns={
            'year': 'Year',
            'coa_iso': 'Country of asylum (ISO)',
            'asylum_seekers': 'UNHCR reported asylum-seeker stock',
        }
    )
)

# A reported zero may mean that the country did not provide this population
# category. It is therefore plotted as unavailable, not as a confirmed nil
# stock.
unhcr_annual_asylum_seeker_stocks['UNHCR stock available'] = (
    unhcr_annual_asylum_seeker_stocks[
        'UNHCR reported asylum-seeker stock'
    ].gt(0)
)
unhcr_annual_asylum_seeker_stocks.loc[
    ~unhcr_annual_asylum_seeker_stocks['UNHCR stock available'],
    'UNHCR reported asylum-seeker stock',
] = np.nan

unhcr_end_2000_asylum_seekers = (
    unhcr_annual_asylum_seeker_stocks.loc[
        unhcr_annual_asylum_seeker_stocks['Year'].eq(2000)
    ]
    .set_index('Country of asylum (ISO)')[
        'UNHCR reported asylum-seeker stock'
    ]
)
top_20_unhcr_end_2000_stock = (
    unhcr_end_2000_asylum_seekers
    .reindex(top_20_worldwide_codes)
    .fillna(0)
)
top_20_unhcr_end_2000_available = top_20_unhcr_end_2000_stock.gt(0)

reported_years = unhcr_annual_asylum_seeker_stocks['Year'].dropna()
print(
    f'UNHCR stock source: {unhcr_stock_cache_source}; '
    f'retrieved {unhcr_stock_retrieved_at:%Y-%m-%d %H:%M UTC}; '
    f'years returned: {reported_years.min()}–{reported_years.max()}; '
    f'positive end-2000 stock for '
    f'{top_20_unhcr_end_2000_available.sum()} of '
    f'{len(top_20_worldwide_codes)} Top 20 countries.'
)

In [ ]:
# %% Load and cache observed EU first-instance pending asylum cases
# Eurostat MIGR_ASYPENCTZM records persons with an asylum application awaiting
# a first-instance decision at the end of each reference month.

EUROSTAT_PENDING_DATASET = 'migr_asypenctzm'
EUROSTAT_PENDING_CACHE_FILE = (
    OBSERVED_DATA_CACHE_DIR / 'eurostat_first_instance_pending.parquet'
)
EUROSTAT_PENDING_METADATA_FILE = (
    OBSERVED_DATA_CACHE_DIR / 'eurostat_first_instance_pending_metadata.json'
)
EU_COUNTRY_TO_EUROSTAT_GEO = {
    'AUT': 'AT', 'BEL': 'BE', 'BGR': 'BG', 'HRV': 'HR', 'CYP': 'CY',
    'CZE': 'CZ', 'DNK': 'DK', 'EST': 'EE', 'FIN': 'FI', 'FRA': 'FR',
    'DEU': 'DE', 'GRC': 'EL', 'HUN': 'HU', 'IRL': 'IE', 'ITA': 'IT',
    'LVA': 'LV', 'LTU': 'LT', 'LUX': 'LU', 'MLT': 'MT', 'NLD': 'NL',
    'POL': 'PL', 'PRT': 'PT', 'ROU': 'RO', 'SVK': 'SK', 'SVN': 'SI',
    'ESP': 'ES', 'SWE': 'SE',
}

euaa_pending_years = np.arange(2008, last_calendar_year + 1)
euaa_pending_periods = [f'{year}-12' for year in euaa_pending_years]
euaa_requested_countries = [
    country_code
    for country_code in top_20_worldwide_codes
    if country_code in EU_COUNTRY_TO_EUROSTAT_GEO
]
euaa_requested_geos = [
    EU_COUNTRY_TO_EUROSTAT_GEO[country_code]
    for country_code in euaa_requested_countries
]

if cache_is_fresh(
        EUROSTAT_PENDING_CACHE_FILE,
        max_age_days=EUROSTAT_CACHE_MAX_AGE_DAYS,
):
    euaa_pending_by_country = pd.read_parquet(
        EUROSTAT_PENDING_CACHE_FILE
    )
    eurostat_pending_cache_source = 'local cache'
    eurostat_pending_retrieved_at = datetime.datetime.fromtimestamp(
        EUROSTAT_PENDING_CACHE_FILE.stat().st_mtime,
        tz=datetime.timezone.utc,
    )
else:
    eurostat_pending_response = requests.get(
        (
            'https://ec.europa.eu/eurostat/api/dissemination/'
            f'statistics/1.0/data/{EUROSTAT_PENDING_DATASET}'
        ),
        params=[
            ('freq', 'M'),
            ('unit', 'PER'),
            ('citizen', 'TOTAL'),
            ('sex', 'T'),
            ('age', 'TOTAL'),
            *[('geo', geo) for geo in euaa_requested_geos],
            *[('time', period) for period in euaa_pending_periods],
        ],
        timeout=60,
    )
    eurostat_pending_response.raise_for_status()
    eurostat_pending_json = eurostat_pending_response.json()

    eurostat_dimensions = eurostat_pending_json['dimension']
    eurostat_sizes = eurostat_pending_json['size']
    eurostat_dimension_names = eurostat_pending_json['id']
    eurostat_values = eurostat_pending_json.get('value', {})

    required_dimension_values = {
        'freq': 'M',
        'unit': 'PER',
        'citizen': 'TOTAL',
        'sex': 'T',
        'age': 'TOTAL',
    }
    for dimension_name, expected_value in required_dimension_values.items():
        returned_values = eurostat_dimensions[dimension_name][
            'category'
        ]['index']
        if expected_value not in returned_values:
            raise ValueError(
                f'Eurostat did not return {dimension_name}={expected_value}. '
                f'Returned: {list(returned_values)}'
            )

    eurostat_geo_index = eurostat_dimensions['geo']['category']['index']
    eurostat_time_index = eurostat_dimensions['time']['category']['index']

    def eurostat_flat_index(dimension_indexes):
        multiplier = 1
        flat_index = 0
        for dimension_name, dimension_size in zip(
                reversed(eurostat_dimension_names),
                reversed(eurostat_sizes),
        ):
            flat_index += dimension_indexes[dimension_name] * multiplier
            multiplier *= dimension_size
        return flat_index

    fixed_dimension_indexes = {
        dimension_name: eurostat_dimensions[dimension_name][
            'category'
        ]['index'][expected_value]
        for dimension_name, expected_value in required_dimension_values.items()
    }
    euaa_pending_rows = []
    for country_code in euaa_requested_countries:
        eurostat_geo = EU_COUNTRY_TO_EUROSTAT_GEO[country_code]
        if eurostat_geo not in eurostat_geo_index:
            continue

        for year, period in zip(euaa_pending_years, euaa_pending_periods):
            if period not in eurostat_time_index:
                continue

            flat_index = eurostat_flat_index(
                {
                    **fixed_dimension_indexes,
                    'geo': eurostat_geo_index[eurostat_geo],
                    'time': eurostat_time_index[period],
                }
            )
            pending_value = eurostat_values.get(str(flat_index))
            if pending_value is not None:
                euaa_pending_rows.append(
                    {
                        'Country of asylum (ISO)': country_code,
                        'Eurostat reporting country': eurostat_geo,
                        'Year': year,
                        'Observed pending cases': pending_value,
                    }
                )

    euaa_pending_by_country = pd.DataFrame(euaa_pending_rows)
    eurostat_pending_retrieved_at = datetime.datetime.now(
        datetime.timezone.utc
    )
    euaa_pending_by_country.to_parquet(
        EUROSTAT_PENDING_CACHE_FILE,
        index=False,
    )
    save_snapshot(
        euaa_pending_by_country,
        'eurostat_first_instance_pending',
        eurostat_pending_retrieved_at,
    )
    save_refresh_metadata(
        EUROSTAT_PENDING_METADATA_FILE,
        {
            'dataset': f'Eurostat {EUROSTAT_PENDING_DATASET}',
            'api_url': (
                'https://ec.europa.eu/eurostat/api/dissemination/'
                f'statistics/1.0/data/{EUROSTAT_PENDING_DATASET}'
            ),
            'retrieved_at_utc': eurostat_pending_retrieved_at.isoformat(),
            'countries_requested': ','.join(euaa_requested_countries),
            'years_requested': f'{euaa_pending_years.min()}–'
                               f'{euaa_pending_years.max()}',
            'row_count': len(euaa_pending_by_country),
        },
    )
    eurostat_pending_cache_source = 'Eurostat API refresh'

required_eurostat_columns = {
    'Country of asylum (ISO)',
    'Eurostat reporting country',
    'Year',
    'Observed pending cases',
}
missing_eurostat_columns = required_eurostat_columns.difference(
    euaa_pending_by_country.columns
)
if missing_eurostat_columns:
    raise ValueError(
        'Eurostat pending-case extract is missing required columns: '
        f'{sorted(missing_eurostat_columns)}'
    )
if euaa_pending_by_country.empty:
    raise ValueError(
        'Eurostat returned no observed first-instance pending-case data '
        'for the EU countries in the worldwide Top 20.'
    )
if euaa_pending_by_country.duplicated(
        ['Country of asylum (ISO)', 'Year']
).any():
    raise ValueError(
        'Eurostat pending-case extract contains duplicate country-year rows.'
    )

euaa_pending_by_country['Observed pending cases'] = pd.to_numeric(
    euaa_pending_by_country['Observed pending cases'],
    errors='raise',
)
latest_eurostat_pending_year = int(euaa_pending_by_country['Year'].max())
reported_eu_countries = set(
    euaa_pending_by_country['Country of asylum (ISO)']
)
missing_eu_countries = sorted(
    set(euaa_requested_countries).difference(reported_eu_countries)
)
if latest_eurostat_pending_year < last_calendar_year:
    print(
        'Warning: Eurostat has not returned December pending-case data for '
        f'{last_calendar_year}; latest returned year is '
        f'{latest_eurostat_pending_year}.'
    )
if missing_eu_countries:
    print(
        'Warning: Eurostat returned no pending-case observations for: '
        f'{", ".join(missing_eu_countries)}.'
    )

print(
    f'Eurostat pending-case source: {eurostat_pending_cache_source}; '
    f'retrieved {eurostat_pending_retrieved_at:%Y-%m-%d %H:%M UTC}; '
    f'years returned: {int(euaa_pending_by_country["Year"].min())}–'
    f'{latest_eurostat_pending_year}; '
    f'countries with observations: {len(reported_eu_countries)}.'
)

In [ ]:
# %% Plot end-2000-anchored application waterfalls with observed stock lines
# The opening stock is UNHCR's reported end-2000 asylum-seeker population.
# Annual flows therefore begin in 2001.  A zero UNHCR value is shown as
# unavailable / zero-reported rather than interpreted as a confirmed nil stock.

facet_columns = 3
facet_rows = int(np.ceil(len(top_20_worldwide_codes) / facet_columns))

fig, axes = plt.subplots(
    facet_rows,
    facet_columns,
    figsize=(facet_columns * 7, facet_rows * 5),
    sharex=False,
)
facet_axes = np.atleast_1d(axes).ravel()

facet_year_ticks = np.unique(
    np.append(
        np.arange(2000, last_calendar_year + 1, 5),
        last_calendar_year,
    )
)

for ax, country_code in zip(facet_axes, top_20_worldwide_codes):
    country_shortfall = top_20_country_shortfall_data.loc[
        top_20_country_shortfall_data[
            "Country of asylum (ISO)"
        ].eq(country_code)
    ].sort_values("Year")

    reported_opening_stock = top_20_unhcr_end_2000_stock.loc[country_code]
    opening_available = top_20_unhcr_end_2000_available.loc[country_code]
    opening_stock = reported_opening_stock if opening_available else 0
    annual_flow_after_2000 = country_shortfall.loc[
        country_shortfall["Year"].gt(2000)
    ].copy()

    opening_balance = (
        opening_stock
        + annual_flow_after_2000["Shortfall"].cumsum().shift(
            fill_value=0
        )
    )
    colors = np.where(
        annual_flow_after_2000["Shortfall"].ge(0),
        "#C53A3A",
        "#00B050",
    )

    if opening_available:
        opening_bar = ax.bar(
            2000,
            opening_stock,
            color="#7030A0",
            width=0.8,
            zorder=3,
        )
        ax.bar_label(
            opening_bar,
            labels=[format_volume_label(opening_stock)],
            padding=2,
            fontsize=7,
        )
        title_anchor = (
            f"UNHCR end-2000: {format_volume_label(opening_stock)}"
        )
    else:
        ax.scatter(
            [2000],
            [0],
            color="#A6A6A6",
            marker="x",
            s=20,
            zorder=5,
        )
        ax.annotate(
            "n/a",
            xy=(2000, 0),
            xytext=(0, 4),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=7,
            color="#666666",
        )
        title_anchor = "UNHCR end-2000: unavailable / zero-reported"

    ax.bar(
        annual_flow_after_2000["Year"],
        annual_flow_after_2000["Shortfall"],
        bottom=opening_balance,
        color=colors,
        width=0.8,
    )
    country_unhcr_stock = (
        unhcr_annual_asylum_seeker_stocks.loc[
            unhcr_annual_asylum_seeker_stocks[
                "Country of asylum (ISO)"
            ].eq(country_code)
        ]
        .dropna(subset=["UNHCR reported asylum-seeker stock"])
        .sort_values("Year")
    )

    if not country_unhcr_stock.empty:
        ax.plot(
            country_unhcr_stock["Year"],
            country_unhcr_stock["UNHCR reported asylum-seeker stock"],
            color="#7030A0",
            linewidth=1.8,
            marker="o",
            markersize=2.8,
            zorder=5,
        )
        latest_unhcr_stock = country_unhcr_stock.iloc[-1]
        ax.annotate(
            format_volume_label(
                latest_unhcr_stock["UNHCR reported asylum-seeker stock"]
            ),
            xy=(
                latest_unhcr_stock["Year"],
                latest_unhcr_stock["UNHCR reported asylum-seeker stock"],
            ),
            xytext=(4, 0),
            textcoords="offset points",
            ha="left",
            va="center",
            fontsize=6.5,
            color="#4A235A",
            fontweight="bold",
            zorder=6,
        )

    country_observed_pending = (
        euaa_pending_by_country.loc[
            euaa_pending_by_country["Country of asylum (ISO)"].eq(
                country_code
            )
        ]
        .sort_values("Year")
    )

    if not country_observed_pending.empty:
        ax.plot(
            country_observed_pending["Year"],
            country_observed_pending["Observed pending cases"],
            color="#5B9BD5",
            linewidth=1.8,
            marker="o",
            markersize=3,
            zorder=5,
        )

        annotation_years = set(
            country_observed_pending.loc[
                country_observed_pending["Year"].mod(4).eq(0),
                "Year",
            ]
        )
        annotation_years.add(country_observed_pending["Year"].iloc[-1])

        for _, observation in country_observed_pending.iterrows():
            if observation["Year"] not in annotation_years:
                continue

            ax.annotate(
                format_volume_label(observation["Observed pending cases"]),
                xy=(
                    observation["Year"],
                    observation["Observed pending cases"],
                ),
                xytext=(0, 4),
                textcoords="offset points",
                ha="center",
                va="bottom",
                fontsize=6.5,
                color="#1F4E79",
                fontweight="bold",
                zorder=6,
            )

    waterfall_ticks = facet_year_ticks
    waterfall_labels = waterfall_ticks.astype(str).tolist()

    ax.axhline(0, color="#000000", linewidth=0.7)
    ax.yaxis.set_major_formatter(EngFormatter(unit="", places=0))
    ax.set_title(
        f"{country_code} — {title_anchor}",
        fontsize=9,
        pad=5,
    )
    ax.set_xticks(waterfall_ticks)
    ax.set_xticklabels(waterfall_labels)
    ax.set_xlim(1999, last_calendar_year + 1)
    ax.tick_params(axis="x", rotation=0, labelsize=8)
    ax.tick_params(axis="y", labelsize=8)
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    ax.set_axisbelow(True)

    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

for ax in facet_axes[len(top_20_worldwide_codes):]:
    ax.set_visible(False)

fig.legend(
    handles=[
        mpl.patches.Patch(
            color="#7030A0",
            label="Observed UNHCR end-2000 stock anchor",
        ),
        mpl.lines.Line2D(
            [], [],
            color="#A6A6A6",
            marker="x",
            linestyle="None",
            label="UNHCR end-2000 stock unavailable / zero-reported",
        ),
        mpl.patches.Patch(
            color="#C53A3A",
            label="Applications exceed decisions",
        ),
        mpl.patches.Patch(
            color="#00B050",
            label="Decisions exceed applications",
        ),
        mpl.lines.Line2D(
            [], [],
            color="#7030A0",
            marker="o",
            linewidth=1.8,
            label="Observed UNHCR end-year asylum-seeker stock",
        ),
        mpl.lines.Line2D(
            [], [],
            color="#5B9BD5",
            marker="o",
            linewidth=1.8,
            label="Observed EU first-instance pending cases",
        ),
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, 0.98),
    ncols=3,
    frameon=False,
    fontsize=8.5,
)
fig.supxlabel("Calendar Year", fontsize=12, y=0.015)
fig.supylabel(
    "Observed end-2000 stock + net annual flow",
    fontsize=12,
    x=0.02,
)
fig.suptitle(
    f"Annual Asylum Application Flow Waterfalls with UNHCR Stocks: "
    f"Top 20 Countries, 2000–{last_calendar_year}",
    fontsize=14,
    y=0.995,
)
fig.text(
    0.5,
    0.943,
    (
        "Annual net flow is applications minus decisions from 2001 onward. "
        "Purple markers are observed UNHCR end-year asylum-seeker stocks; blue "
        "markers are observed December EU first-instance pending cases. Where "
        "UNHCR reports zero in 2000, the panel is "
        "unanchored; the annual bars show net flow, not a calculated backlog."
    ),
    ha="center",
    fontsize=8,
    color="#4A235A",
)
fig.tight_layout(rect=(0.01, 0.03, 0.99, 0.925))
fig.subplots_adjust(wspace=0.2, hspace=0.3)
plt.show()

In [ ]:
# %% Observed annual pending asylum stocks: UNHCR Top 20 countries
# Purple = UNHCR end-year asylum-seeker stock. Blue = Eurostat / EUAA
# first-instance pending cases for EU countries, available from 2008 onward.

facet_columns = 3
facet_rows = int(np.ceil(len(top_20_worldwide_codes) / facet_columns))
fig, axes = plt.subplots(
    facet_rows,
    facet_columns,
    figsize=(facet_columns * 7, facet_rows * 4.5),
    sharex=False,
    sharey=False,
)
facet_axes = np.atleast_1d(axes).ravel()

annual_stock_ticks = np.unique(
    np.append(
        np.arange(2000, last_calendar_year + 1, 5),
        last_calendar_year,
    )
)

for ax, country_code in zip(facet_axes, top_20_worldwide_codes):
    country_unhcr_stock = (
        unhcr_annual_asylum_seeker_stocks.loc[
            unhcr_annual_asylum_seeker_stocks[
                "Country of asylum (ISO)"
            ].eq(country_code)
        ]
        .dropna(subset=["UNHCR reported asylum-seeker stock"])
        .sort_values("Year")
    )
    country_euaa_stock = (
        euaa_pending_by_country.loc[
            euaa_pending_by_country[
                "Country of asylum (ISO)"
            ].eq(country_code)
        ]
        .sort_values("Year")
    )

    y_candidates = []
    if not country_unhcr_stock.empty:
        y_candidates.append(
            country_unhcr_stock["UNHCR reported asylum-seeker stock"].max()
        )
    if not country_euaa_stock.empty:
        y_candidates.append(
            country_euaa_stock["Observed pending cases"].max()
        )
    country_ymax = max(y_candidates) if y_candidates else 1
    country_y_limit = country_ymax * 1.15 if country_ymax > 0 else 1

    if not country_unhcr_stock.empty:
        ax.plot(
            country_unhcr_stock["Year"],
            country_unhcr_stock["UNHCR reported asylum-seeker stock"],
            color="#7030A0",
            linewidth=2.2,
            marker="o",
            markersize=3.5,
            label="UNHCR end-year asylum-seeker stock",
        )
        latest_unhcr_stock = country_unhcr_stock.iloc[-1]
        ax.annotate(
            format_volume_label(
                latest_unhcr_stock["UNHCR reported asylum-seeker stock"]
            ),
            xy=(
                latest_unhcr_stock["Year"],
                latest_unhcr_stock["UNHCR reported asylum-seeker stock"],
            ),
            xytext=(0, 5),
            textcoords="offset points",
            ha="center",
            va="bottom",
            fontsize=7,
            color="#4A235A",
            fontweight="bold",
        )
    else:
        ax.text(
            0.5,
            0.5,
            "No positive UNHCR\nstock reported",
            transform=ax.transAxes,
            ha="center",
            va="center",
            fontsize=9,
            color="#666666",
        )

    if not country_euaa_stock.empty:
        ax.plot(
            country_euaa_stock["Year"],
            country_euaa_stock["Observed pending cases"],
            color="#5B9BD5",
            linewidth=1.8,
            marker="o",
            markersize=3,
            label="EUAA / Eurostat first-instance pending cases",
        )
        latest_euaa_stock = country_euaa_stock.iloc[-1]
        ax.annotate(
            format_volume_label(
                latest_euaa_stock["Observed pending cases"]
            ),
            xy=(
                latest_euaa_stock["Year"],
                latest_euaa_stock["Observed pending cases"],
            ),
            xytext=(0, -10),
            textcoords="offset points",
            ha="center",
            va="top",
            fontsize=7,
            color="#1F4E79",
            fontweight="bold",
        )

    ax.set_title(country_code, fontsize=11, pad=5)
    ax.set_ylim(0, country_y_limit)
    ax.yaxis.set_major_formatter(EngFormatter(unit="", places=0))
    ax.set_xticks(annual_stock_ticks)
    ax.tick_params(axis="x", rotation=0, labelsize=8)
    ax.tick_params(axis="y", labelsize=8)
    ax.grid(axis="y", linestyle="--", alpha=0.3)
    ax.set_axisbelow(True)

    for spine in ("top", "right"):
        ax.spines[spine].set_visible(False)

for ax in facet_axes[len(top_20_worldwide_codes):]:
    ax.set_visible(False)

fig.legend(
    handles=[
        mpl.lines.Line2D(
            [], [],
            color="#7030A0",
            marker="o",
            linewidth=2.2,
            label="UNHCR end-year asylum-seeker stock",
        ),
        mpl.lines.Line2D(
            [], [],
            color="#5B9BD5",
            marker="o",
            linewidth=1.8,
            label="EUAA / Eurostat first-instance pending cases",
        ),
    ],
    loc="upper center",
    bbox_to_anchor=(0.5, 0.978),
    ncols=2,
    frameon=False,
    fontsize=9,
)
fig.suptitle(
    f"Observed Annual Asylum-Seeker Stocks: Top 20 Receiving Countries, "
    f"2000–{last_calendar_year}",
    fontsize=14,
    y=0.995,
)
fig.text(
    0.5,
    0.955,
    (
        "UNHCR values are end-year asylum-seeker stocks. Eurostat / EUAA "
        "values are December first-instance pending cases. Their definitions "
        "differ, so the series are compared for context, not combined. "
        "Each panel uses its own y-axis scale."
    ),
    ha="center",
    fontsize=8,
    color="#4A235A",
)
fig.supxlabel("Calendar Year", fontsize=11, y=0.015)
fig.supylabel("Observed pending asylum stock", fontsize=11, x=0.02)
fig.tight_layout(rect=(0.03, 0.04, 0.99, 0.93))
plt.show()

In [ ]:
# %% Last Calendar Year - Prepare Datasets
last_calendar_year_data = worldwide_annual_data.loc[
    worldwide_annual_data["Year"].eq(last_calendar_year)
].iloc[0]

last_calendar_year_applications = last_calendar_year_data["applied"]
last_calendar_year_decisions = last_calendar_year_data["Total decisions"]
last_calendar_year_shortfall = (
    last_calendar_year_applications - last_calendar_year_decisions
)

Last Calendar Year - Identify Top 20 Receiving Countries by Application Volume

In [ ]:
# %% Last calendar year: identify the Top 20 asylum countries by application volume
# Keep reported persons and cases separate; use the combined figure only as a
# descriptive ranking measure.
last_calendar_year_country_applications = (
    UNTotalAppsDecs.loc[
        UNTotalAppsDecs['Year'].eq(last_calendar_year)
    ]
    .groupby(
        ['Country of asylum (ISO)', 'Cases / Persons'],
        as_index=False,
    )['applied']
    .sum()
    .pivot(
        index='Country of asylum (ISO)',
        columns='Cases / Persons',
        values='applied',
    )
    .reindex(columns=['P', 'C'], fill_value=0)
    .fillna(0)
    .rename(columns={
        'P': 'Applications reported as persons',
        'C': 'Applications reported as cases',
    })
)

last_calendar_year_country_applications[
    'Applications (mixed reporting units)'
] = (
    last_calendar_year_country_applications[
        'Applications reported as persons'
    ]
    + last_calendar_year_country_applications[
        'Applications reported as cases'
    ]
)

top_20_last_calendar_year_countries = (
    last_calendar_year_country_applications
    .reset_index()
    .nlargest(20, 'Applications (mixed reporting units)')[
        'Country of asylum (ISO)'
    ]
)

Last Calendar Year - Show the Top 20 Receiving Countries by Application Volume

In [ ]:
# %% Last calendar year: display Top 20 asylum countries by application volume
top_20_last_calendar_year_display = (
    last_calendar_year_country_applications
    .reset_index()
    .loc[
        lambda dataframe: dataframe['Country of asylum (ISO)'].isin(
            top_20_last_calendar_year_countries
        )
    ]
    .rename(columns={
        'Country of asylum (ISO)': 'ISO country code',
    })
    .merge(
        iso_country_names.set_index('ISO country code')[['Country name']],
        left_on='ISO country code',
        right_index=True,
        how='left',
    )
    [[
        'ISO country code',
        'Country name',
        'Applications reported as persons',
        'Applications reported as cases',
        'Applications (mixed reporting units)',
    ]]
    .sort_values('Applications (mixed reporting units)', ascending=False)
    .reset_index(drop=True)
)

top_20_last_calendar_year_display.style.format({
    'Applications reported as persons': '{:,.0f}',
    'Applications reported as cases': '{:,.0f}',
    'Applications (mixed reporting units)': '{:,.0f}',
}).hide(axis='index')

Last Calendar Year - Plot Top 20 Countries Share of Applications on a Pie Chart

In [ ]:
# %% Last calendar year: Top 20 share of reported application volume
# Persons and cases are intentionally combined here for a descriptive,
# mixed-unit comparison matching the Top-20 country ranking.
top_20_last_calendar_year_apps = (
    last_calendar_year_country_applications
    .reindex(top_20_last_calendar_year_countries)[
        'Applications (mixed reporting units)'
    ]
    .sum()
)
total_last_calendar_year_apps = (
    last_calendar_year_country_applications[
        'Applications (mixed reporting units)'
    ]
    .sum()
)
other_last_calendar_year_apps = (
    total_last_calendar_year_apps - top_20_last_calendar_year_apps
)

countries = ['Top 20 Countries', 'All Other Countries']
data = [top_20_last_calendar_year_apps, other_last_calendar_year_apps]
colors = ('#002664', '#7A7A7A')

fig, ax = plt.subplots(figsize=(10, 8))
wedges, texts, autotexts = ax.pie(
    data,
    labels=None,
    autopct=lambda pct: format_pie_slice(pct, data),
    startangle=165,
    textprops=dict(color='#FFFFFF', fontsize=12),
    colors=colors,
    wedgeprops={'edgecolor': 'w', 'linewidth': 2},
)
ax.legend(
    wedges,
    countries,
    loc='upper center',
    bbox_to_anchor=(0.5, 1.0),
    ncols=2,
    frameon=False,
)

fig.subplots_adjust(top=0.82)
ax.set_title(
    f'Application Volume (Persons and Cases Combined): '
    f'Top 20 vs All Other Countries, {last_calendar_year}',
    fontsize=12,
    pad=36,
)
print('-' * 100)
plt.show()
print('-' * 100)

In [ ]:
# %% Last calendar year: prepare Top-20 chart datasets
# Aggregate persons and cases within each selected country to preserve the
# existing mixed-unit chart convention.
top_20_last_calendar_year_apps = (
    UNTotalAppsDecs.loc[
        UNTotalAppsDecs['Year'].eq(last_calendar_year)
        & UNTotalAppsDecs['Country of asylum (ISO)'].isin(
            top_20_last_calendar_year_countries
        )
    ]
    .groupby('Country of asylum (ISO)', as_index=False)[
        ['applied', 'Total decisions']
    ]
    .sum()
    .sort_values('applied', ascending=False)
    .reset_index(drop=True)
)

top_20_last_calendar_year_apps['FormattedApps'] = (
    top_20_last_calendar_year_apps['applied'].map('{:,.0f}'.format)
)

total_top_20_last_calendar_year_apps = (
    top_20_last_calendar_year_apps['applied'].sum()
)
total_top_20_last_calendar_year_decisions = (
    top_20_last_calendar_year_apps['Total decisions'].sum()
)
top_20_last_calendar_year_shortfall = (
    total_top_20_last_calendar_year_apps
    - total_top_20_last_calendar_year_decisions
)

Last Calendar Year - Plot Top 20 Receiving Country Application Volumes as a Bar Chart

In [ ]:
# %% Last calendar year: plot Top 20 application volumes
# Rank by the combined reported persons-and-cases application volume.
last_calendar_year_chart_data = (
    top_20_last_calendar_year_apps
    .sort_values('applied')
)
bar_colors = [
    '#732282' if country_code == 'GBR' else '#002664'
    for country_code in
    last_calendar_year_chart_data['Country of asylum (ISO)']
]

fig, ax = plt.subplots(figsize=(16, 9))
bars = ax.barh(
    last_calendar_year_chart_data['Country of asylum (ISO)'],
    last_calendar_year_chart_data['applied'],
    color=bar_colors,
)
ax.bar_label(
    bars,
    labels=last_calendar_year_chart_data['FormattedApps'],
    padding=4,
    fontsize=10,
)
ax.xaxis.set_major_formatter(EngFormatter(unit='', places=0))
ax.set_xlabel(
    'Reported application volume (persons and cases combined)',
    fontsize=12,
)
ax.set_ylabel('Country of asylum (ISO)', fontsize=12)
ax.set_title(
    f'Top 20 Countries by Application Volume '
    f'(Persons and Cases Combined), {last_calendar_year}',
    fontsize=12,
    pad=16,
)
ax.grid(axis='x', linestyle='--', alpha=0.35)
ax.set_axisbelow(True)
ax.tick_params(axis='y', length=0)
for spine in ('top', 'right', 'left'):
    ax.spines[spine].set_visible(False)
fig.tight_layout()
print('-' * 100)
plt.show()
print('-' * 100)

Last Calendar Year - Plot Top 20 Receiving Countries Applications minus Decisions on a Bar Chart

Individual Countries Analysis

Individual Countries: Identify Top 20 Receiving Countries by Application Volume since 2000

In [ ]:
# %% Identify annual applications and decision outcomes for each Top 20 country
top_20_country_outcome_data = (
    UNTotalAppsDecs.loc[
        UNTotalAppsDecs['Year'].between(2000, last_calendar_year)
        & UNTotalAppsDecs['Country of asylum (ISO)'].isin(
            top_20_worldwide_codes
        )
    ]
    .groupby(['Year', 'Country of asylum (ISO)'], as_index=False)[
        [
            'applied',
            'Recognized decisions',
            'Complementary protection',
            'Rejected decisions',
            'Otherwise closed',
        ]
    ]
    .sum()
)
top_20_country_outcome_data['Total decisions'] = (
    top_20_country_outcome_data[worldwide_outcome_columns].sum(axis=1)
)

# Add the observed UNHCR end-year asylum-seeker stock where it is reported.
# This is a population stock (persons), not an estimate of 1999 applications.
top_20_country_outcome_data = top_20_country_outcome_data.merge(
    unhcr_annual_asylum_seeker_stocks[
        [
            'Year',
            'Country of asylum (ISO)',
            'UNHCR reported asylum-seeker stock',
        ]
    ],
    on=['Year', 'Country of asylum (ISO)'],
    how='left',
)

Individual Countries - Plot Annual Applications and Decision Outcomes for each Top 20 Country on Vertical Bar and
Line Combination Charts

In [ ]:
# %% Plot annual applications, decisions, and observed UNHCR stock for each Top 20 country
for country_code in top_20_worldwide_codes:
    country_outcomes = (
        top_20_country_outcome_data.loc[
            top_20_country_outcome_data['Country of asylum (ISO)'].eq(
                country_code
            )
        ]
        .sort_values('Year')
    )

    country_total_decisions = country_outcomes['Total decisions']
    country_unhcr_stock = country_outcomes.dropna(
        subset=['UNHCR reported asylum-seeker stock']
    )
    y_max_candidates = [
        country_total_decisions.max(),
        country_outcomes['applied'].max(),
    ]
    if not country_unhcr_stock.empty:
        y_max_candidates.append(
            country_unhcr_stock['UNHCR reported asylum-seeker stock'].max()
        )
    common_y_max = max(y_max_candidates)
    common_y_limit = common_y_max * 1.10 if common_y_max > 0 else 1

    fig, ax = plt.subplots(figsize=(16, 9))

    # Labels remain inside their own bar segment only when there is enough
    # vertical space for an 8-point label.
    minimum_annotation_height = common_y_limit * 0.055
    bottom = np.zeros(len(country_outcomes))
    for column, label, color in zip(
            worldwide_outcome_columns,
            worldwide_outcome_labels,
            worldwide_outcome_colors,
    ):
        values = country_outcomes[column]
        bars = ax.bar(
            country_outcomes['Year'],
            values,
            bottom=bottom,
            label=label,
            color=color,
            width=0.8,
        )
        ax.bar_label(
            bars,
            labels=[
                format_volume_label(value)
                if value >= minimum_annotation_height else ''
                for value in values
            ],
            label_type='center',
            fontsize=8,
            color=(
                '#000000'
                if column in (
                    'Recognized decisions',
                    'Complementary protection',
                )
                else 'white'
            ),
        )
        bottom += values.to_numpy()

    # Total decisions: annotate only bars that have enough visual space.
    minimum_total_annotation_height = common_y_limit * 0.08
    for year, total_decisions in zip(
            country_outcomes['Year'],
            country_total_decisions,
    ):
        if total_decisions >= minimum_total_annotation_height:
            ax.annotate(
                format_volume_label(total_decisions),
                xy=(year, total_decisions),
                xytext=(0, 3),
                textcoords='offset points',
                ha='center',
                va='bottom',
                fontsize=8,
                color='#333333',
                fontweight='bold',
            )

    applications_axis = ax.twinx()
    applications_line, = applications_axis.plot(
        country_outcomes['Year'],
        country_outcomes['applied'],
        color='#002664',
        linewidth=2.5,
        linestyle=':',
        marker='o',
        markersize=4,
        label='Applications',
    )

    # Label only the highest application volumes to avoid clutter.
    minimum_application_annotation_height = common_y_limit * 0.65
    for year, applications in zip(
            country_outcomes['Year'],
            country_outcomes['applied'],
    ):
        if applications >= minimum_application_annotation_height:
            applications_axis.annotate(
                format_volume_label(applications),
                xy=(year, applications),
                xytext=(0, 7),
                textcoords='offset points',
                ha='center',
                va='bottom',
                fontsize=8,
                color='#002664',
                fontweight='bold',
            )

    # Observed stock is shown separately from the annual application flow.
    if not country_unhcr_stock.empty:
        applications_axis.plot(
            country_unhcr_stock['Year'],
            country_unhcr_stock['UNHCR reported asylum-seeker stock'],
            color='#7030A0',
            linewidth=2.0,
            marker='o',
            markersize=3.5,
            zorder=5,
        )
        latest_unhcr_stock = country_unhcr_stock.iloc[-1]
        applications_axis.annotate(
            format_volume_label(
                latest_unhcr_stock['UNHCR reported asylum-seeker stock']
            ),
            xy=(
                latest_unhcr_stock['Year'],
                latest_unhcr_stock['UNHCR reported asylum-seeker stock'],
            ),
            xytext=(4, 0),
            textcoords='offset points',
            ha='left',
            va='center',
            fontsize=7,
            color='#4A235A',
            fontweight='bold',
        )

    # Keep both y-axes on the same numeric scale.
    ax.set_ylim(0, common_y_limit)
    applications_axis.set_ylim(0, common_y_limit)
    applications_axis.yaxis.set_major_formatter(
        EngFormatter(unit='', places=0)
    )
    applications_axis.set_ylabel(
        'Reported applications / asylum-seeker stock',
        fontsize=12,
    )
    applications_axis.grid(False)
    applications_axis.spines['top'].set_visible(False)

    ax.yaxis.set_major_formatter(EngFormatter(unit='', places=0))
    ax.set_xlabel('Calendar Year', fontsize=12)
    ax.set_ylabel('Number of decisions', fontsize=12)
    ax.set_title(
        f'Asylum Decisions, Applications, and UNHCR Stock: {country_code}, '
        f'2000–{last_calendar_year}',
        fontsize=16,
        pad=16,
    )
    ax.text(
        0.5,
        .995,
        'Purple line: observed UNHCR end-year asylum-seeker stock (persons), '
        'not an application flow. No usable country-level 1999 stock is available.',
        transform=ax.transAxes,
        ha='center',
        va='bottom',
        fontsize=9,
        color='#4A235A',
    )
    ax.set_xticks(country_outcomes['Year'])
    ax.tick_params(axis='x', rotation=0)
    ax.grid(axis='y', linestyle='--', alpha=0.35)
    ax.set_axisbelow(True)
    decision_handles, decision_labels = ax.get_legend_handles_labels()
    unhcr_stock_handle = mpl.lines.Line2D(
        [], [],
        color='#7030A0',
        marker='o',
        linewidth=2.0,
        label='UNHCR end-year asylum-seeker stock',
    )
    ax.legend(
        decision_handles + [applications_line, unhcr_stock_handle],
        decision_labels + [
            'Applications',
            'UNHCR end-year asylum-seeker stock',
        ],
        loc='upper center',
        bbox_to_anchor=(0.5, 1.0),
        ncols=3,
        frameon=False,
    )
    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)
    fig.tight_layout()
    plt.show()

In [ ]:
# %% Faceted 100%-stacked decision-outcome composition for Top 20 countries
# Three-year rolling decision totals make composition trends readable without
# overlapping outcome-rate lines.

OUTCOME_RATE_ROLLING_WINDOW = 3
MINIMUM_ROLLING_DECISIONS = 150
outcome_rate_columns = {
    'Recognized decisions': 'Granted',
    'Complementary protection': 'Other protection',
    'Rejected decisions': 'Refused',
    'Otherwise closed': 'Closed',
}
outcome_rate_colours = {
    'Granted': '#00B050',
    'Other protection': '#FFD700',
    'Refused': '#C53A3A',
    'Closed': '#7A7A7A',
}

facet_columns = 3
facet_rows = int(np.ceil(len(top_20_worldwide_codes) / facet_columns))
fig, axes = plt.subplots(
    facet_rows,
    facet_columns,
    figsize=(facet_columns * 7, facet_rows * 3.4),
    sharex=True,
    sharey=True,
)
facet_axes = np.atleast_1d(axes).ravel()
outcome_rate_year_ticks = np.unique(
    np.append(
        np.arange(2000, last_calendar_year + 1, 5),
        last_calendar_year,
    )
)

for ax, country_code in zip(facet_axes, top_20_worldwide_codes):
    country_outcomes = (
        top_20_country_outcome_data.loc[
            top_20_country_outcome_data['Country of asylum (ISO)'].eq(
                country_code
            ),
            ['Year', 'Total decisions', *outcome_rate_columns],
        ]
        .sort_values('Year')
        .set_index('Year')
    )

    rolling_outcomes = country_outcomes.rolling(
        window=OUTCOME_RATE_ROLLING_WINDOW,
        center=True,
        min_periods=OUTCOME_RATE_ROLLING_WINDOW,
    ).sum()
    valid_years = rolling_outcomes['Total decisions'].ge(
        MINIMUM_ROLLING_DECISIONS
    )
    rolling_rates = rolling_outcomes.loc[valid_years].copy()
    for source_column, outcome_label in outcome_rate_columns.items():
        rolling_rates[outcome_label] = (
            rolling_rates[source_column]
            .div(rolling_rates['Total decisions'])
        )

    bottom = np.zeros(len(rolling_rates))
    for outcome_label, colour in outcome_rate_colours.items():
        values = rolling_rates[outcome_label].to_numpy()
        ax.bar(
            rolling_rates.index,
            values,
            bottom=bottom,
            width=0.85,
            color=colour,
            edgecolor='white',
            linewidth=0.25,
        )
        bottom += values

    ax.set_title(country_code, fontsize=10, pad=5)
    ax.set_ylim(0, 1)
    ax.set_xticks(outcome_rate_year_ticks)
    ax.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    ax.tick_params(axis='x', rotation=0, labelsize=8)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)

    for spine in ('top', 'right'):
        ax.spines[spine].set_visible(False)

for ax in facet_axes[len(top_20_worldwide_codes):]:
    ax.set_visible(False)

fig.legend(
    handles=[
        mpl.patches.Patch(color=colour, label=outcome_label)
        for outcome_label, colour in outcome_rate_colours.items()
    ],
    loc='upper center',
    bbox_to_anchor=(0.5, 0.982),
    ncols=4,
    frameon=False,
    fontsize=9,
)
fig.suptitle(
    f'Decision-Outcome Composition: Top 20 Receiving Countries, '
    f'2000–{last_calendar_year}',
    fontsize=14,
    y=0.997,
)
fig.text(
    0.5,
    0.977,
    (
        f'Each column is a {OUTCOME_RATE_ROLLING_WINDOW}-year centred rolling '
        'share of total decisions; only windows with at least '
        f'{MINIMUM_ROLLING_DECISIONS:,} decisions are shown.'
    ),
    ha='center',
    fontsize=8,
    color='#4A235A',
)
fig.supxlabel('Calendar Year', fontsize=11, y=0.012)
fig.supylabel('Share of total decisions', fontsize=11, x=0.015)
fig.tight_layout(rect=(0.02, 0.03, 0.99, 0.955))
fig.subplots_adjust(wspace=0.18, hspace=0.32)
plt.show()

Top 20 Receiving Countries from 2000 - Calculate Face-Value Productivity (FVP)

Individual Countries - Calculate Annual Minimum Shortfall for each Top 20 Receiving Country Last Calendar Year

Individual Countries - Plot Minimum Backlog Last Calendar Year for Top 20 Receiving Countries since 2000

In [ ]:
# %% Individual Countries - Plot Annual Decisions minus Applications for each Top 20 Receiving Country since 2000
facet_columns = 3
facet_rows = int(np.ceil(len(top_20_worldwide_codes) / facet_columns))
facet_year_ticks = np.unique(
    np.append(np.arange(2000, last_calendar_year + 1, 5), last_calendar_year)
)
fig, axes = plt.subplots(
    facet_rows, facet_columns, figsize=(facet_columns * 7, facet_rows * 5),
    sharex=False
)
facet_axes = np.atleast_1d(axes).ravel()

for ax, country_code in zip(facet_axes, top_20_worldwide_codes):
    country_shortfall = top_20_country_shortfall_data.loc[
        top_20_country_shortfall_data[
            'Country of asylum (ISO)'] == country_code
        ]
    colors = np.where(country_shortfall['Shortfall'] >= 0, '#C53A3A',
                      '#00B050')
    ax.bar(country_shortfall['Year'], country_shortfall['Shortfall'],
           color=colors, width=0.8)
    ax.axhline(0, color='#000000', linewidth=0.7)
    ax.yaxis.set_major_formatter(EngFormatter(unit = '', places = 0))
    ax.set_title(country_code, fontsize=10, pad=5)
    ax.set_xticks(facet_year_ticks)
    ax.tick_params(axis='x', rotation=0, labelsize=8)
    ax.tick_params(axis='y', labelsize=8)
    ax.grid(axis='y', linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    for spine in ('top', 'right', 'bottom', 'left'):
        ax.spines[spine].set_visible(False)
for ax in facet_axes[len(top_20_worldwide_codes):]:
    ax.set_visible(False)
fig.legend(
    handles=[
        mpl.patches.Patch(color='#C53A3A',
                          label='Applications exceed decisions'),
        mpl.patches.Patch(color='#00B050',
                          label='Decisions exceed applications'),
    ],
    loc='upper center', bbox_to_anchor=(0.5, 0.985), ncols=2, frameon=False,
)
fig.supxlabel('Calendar Year', fontsize=10, y=0.015)
fig.supylabel('Annual shortfall (applications minus decisions)', fontsize=10, x = 0.01)
fig.suptitle(
    f'Annual Asylum Application Shortfall: Top 20 Countries, 20'
    f'00–{last_calendar_year}',
    fontsize=12, y=0.995,
)
# Adjust spacing using the Figure method
fig.tight_layout(rect=(0.01, 0.03, .99, 0.97)) # Left, bottom, right, top
fig.subplots_adjust(wspace=0.2, hspace=0.3)  # horizontal & vertical space
plt.show()

Individual Countries - Plot Annual Shortfall for each Top 20 Receiving Country from 2000

Individual Countries - Plot Annual Shortfall for each Top 20 Receiving Countries from 2000 as Waterfall Charts

In [ ]:
# %% Interactive receiving-country dashboard
# Requires an active Jupyter kernel: choose a country to redraw its charts.

COUNTRY_DASHBOARD_ROLLING_WINDOW = 3
COUNTRY_DASHBOARD_MINIMUM_DECISIONS = 150
country_dashboard_outcomes = {
    'Recognized decisions': 'Granted',
    'Complementary protection': 'Other protection',
    'Rejected decisions': 'Refused',
    'Otherwise closed': 'Closed',
}
country_dashboard_colours = {
    'Granted': '#00B050',
    'Other protection': '#FFD700',
    'Refused': '#C53A3A',
    'Closed': '#7A7A7A',
}

country_dashboard_codes = sorted(
    UNTotalAppsDecs.loc[
        UNTotalAppsDecs['Year'].between(2000, last_calendar_year),
        'Country of asylum (ISO)',
    ]
    .dropna()
    .astype(str)
    .loc[lambda codes: codes.str.fullmatch(r'[A-Z]{3}')]
    .unique()
)

def country_dashboard_label(country_code):
    country = pycountry.countries.get(alpha_3=country_code)
    country_name = country.name if country is not None else country_code
    return f'{country_code} — {country_name}'

country_dashboard_options = [
    (country_dashboard_label(country_code), country_code)
    for country_code in country_dashboard_codes
]
country_dashboard_default = (
    'AUS' if 'AUS' in country_dashboard_codes
    else country_dashboard_codes[0]
)

country_dashboard_selector = widgets.Dropdown(
    options=country_dashboard_options,
    value=country_dashboard_default,
    description='Receiving country:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='360px'),
)
country_dashboard_output = widgets.Output()

def render_country_dashboard(change=None):
    country_code = country_dashboard_selector.value

    with country_dashboard_output:
        clear_output(wait=True)

        country_data = (
            UNTotalAppsDecs.loc[
                UNTotalAppsDecs['Year'].between(2000, last_calendar_year)
                & UNTotalAppsDecs['Country of asylum (ISO)'].eq(
                    country_code
                )
            ]
            .groupby('Year', as_index=False)[
                [
                    'applied',
                    'Recognized decisions',
                    'Complementary protection',
                    'Rejected decisions',
                    'Otherwise closed',
                ]
            ]
            .sum()
            .sort_values('Year')
        )

        if country_data.empty:
            print(f'No annual decision data are available for {country_code}.')
            return

        country_data['Total decisions'] = country_data[
            list(country_dashboard_outcomes)
        ].sum(axis=1)

        country_stock = pd.DataFrame()
        if 'unhcr_annual_asylum_seeker_stocks' in globals():
            country_stock = (
                unhcr_annual_asylum_seeker_stocks.loc[
                    unhcr_annual_asylum_seeker_stocks[
                        'Country of asylum (ISO)'
                    ].eq(country_code)
                ]
                .dropna(subset=['UNHCR reported asylum-seeker stock'])
                .sort_values('Year')
            )

        y_max_candidates = [
            country_data['Total decisions'].max(),
            country_data['applied'].max(),
        ]
        if not country_stock.empty:
            y_max_candidates.append(
                country_stock['UNHCR reported asylum-seeker stock'].max()
            )
        common_y_limit = max(y_max_candidates) * 1.10
        if common_y_limit <= 0:
            common_y_limit = 1

        fig, (ax_flow, ax_composition) = plt.subplots(
            2,
            1,
            figsize=(16, 14),
            gridspec_kw={'height_ratios': [1.15, 1]},
        )

        decision_bottom = np.zeros(len(country_data))
        for source_column, outcome_label in country_dashboard_outcomes.items():
            ax_flow.bar(
                country_data['Year'],
                country_data[source_column],
                bottom=decision_bottom,
                width=0.8,
                color=country_dashboard_colours[outcome_label],
                edgecolor='white',
                linewidth=0.35,
                label=outcome_label,
            )
            decision_bottom += country_data[source_column].to_numpy()

        applications_axis = ax_flow.twinx()
        applications_line, = applications_axis.plot(
            country_data['Year'],
            country_data['applied'],
            color='#002664',
            linewidth=2.5,
            linestyle=':',
            marker='o',
            markersize=4,
            label='Applications',
        )

        stock_handle = None
        if not country_stock.empty:
            applications_axis.plot(
                country_stock['Year'],
                country_stock['UNHCR reported asylum-seeker stock'],
                color='#7030A0',
                linewidth=2.0,
                marker='o',
                markersize=3.5,
            )
            stock_handle = mpl.lines.Line2D(
                [], [],
                color='#7030A0',
                marker='o',
                linewidth=2.0,
                label='UNHCR end-year asylum-seeker stock',
            )

        ax_flow.set_ylim(0, common_y_limit)
        applications_axis.set_ylim(0, common_y_limit)

        # Label the latest annual flow values only where the annotation is
        # large enough to remain legible within the shared y-axis range.
        latest_flow = country_data.iloc[-1]
        latest_year = latest_flow['Year']
        latest_decisions = latest_flow['Total decisions']
        latest_applications = latest_flow['applied']
        minimum_labelled_volume = common_y_limit * 0.04

        if latest_decisions >= minimum_labelled_volume:
            decision_label_inside = latest_decisions >= common_y_limit * 0.91
            ax_flow.annotate(
                f'{format_volume_label(latest_decisions)} decisions',
                xy=(latest_year, latest_decisions),
                xytext=(0, -5 if decision_label_inside else 5),
                textcoords='offset points',
                ha='center',
                va='top' if decision_label_inside else 'bottom',
                fontsize=8,
                color='white' if decision_label_inside else '#1F1F1F',
            )

        if latest_applications >= minimum_labelled_volume:
            application_label_inside = (
                latest_applications >= common_y_limit * 0.91
            )
            applications_axis.annotate(
                f'{format_volume_label(latest_applications)} applications',
                xy=(latest_year, latest_applications),
                xytext=(0, -7 if application_label_inside else 7),
                textcoords='offset points',
                ha='center',
                va='top' if application_label_inside else 'bottom',
                fontsize=8,
                color='#002664',
            )

        if not country_stock.empty:
            latest_stock = country_stock.iloc[-1]
            latest_stock_year = latest_stock['Year']
            latest_stock_value = latest_stock[
                'UNHCR reported asylum-seeker stock'
            ]
            if latest_stock_value >= minimum_labelled_volume:
                shares_latest_year = latest_stock_year == latest_year
                near_application = (
                    shares_latest_year
                    and abs(latest_stock_value - latest_applications)
                    <= common_y_limit * 0.12
                )
                stock_label_inside = (
                    latest_stock_value >= common_y_limit * 0.91
                    or near_application
                )
                applications_axis.annotate(
                    f'{format_volume_label(latest_stock_value)} stock',
                    xy=(latest_stock_year, latest_stock_value),
                    xytext=(0, -19 if stock_label_inside else 7),
                    textcoords='offset points',
                    ha='center',
                    va='top' if stock_label_inside else 'bottom',
                    fontsize=8,
                    color='#7030A0',
                )

        ax_flow.yaxis.set_major_formatter(EngFormatter(unit='', places=0))
        applications_axis.yaxis.set_major_formatter(
            EngFormatter(unit='', places=0)
        )
        ax_flow.set_ylabel('Number of decisions', fontsize=11)
        ax_flow.set_title(
            'Annual reported applications and decisions',
            loc='left',
            fontsize=12,
            pad=10,
        )
        applications_axis.set_ylabel(
            'Reported applications / asylum-seeker stock',
            fontsize=11,
        )
        ax_flow.grid(axis='y', linestyle='--', alpha=0.35)
        ax_flow.set_axisbelow(True)
        applications_axis.grid(False)
        applications_axis.spines['top'].set_visible(False)
        ax_flow.set_xticks(
            np.unique(
                np.append(
                    np.arange(2000, last_calendar_year + 1, 2),
                    last_calendar_year,
                )
            )
        )
        ax_flow.tick_params(axis='x', rotation=0, labelbottom=True)
        for spine in ('top', 'right'):
            ax_flow.spines[spine].set_visible(False)

        composition_data = country_data[
            ['Year', 'Total decisions', *country_dashboard_outcomes]
        ].set_index('Year')
        rolling_outcomes = composition_data.rolling(
            window=COUNTRY_DASHBOARD_ROLLING_WINDOW,
            min_periods=1,
        ).sum()
        rolling_rates = rolling_outcomes.loc[
            rolling_outcomes['Total decisions'].ge(
                COUNTRY_DASHBOARD_MINIMUM_DECISIONS
            )
        ].copy()
        for source_column, outcome_label in country_dashboard_outcomes.items():
            rolling_rates[outcome_label] = (
                rolling_rates[source_column]
                .div(rolling_rates['Total decisions'])
            )

        composition_bottom = np.zeros(len(rolling_rates))
        for outcome_label, colour in country_dashboard_colours.items():
            values = rolling_rates[outcome_label].to_numpy()
            ax_composition.bar(
                rolling_rates.index,
                values,
                bottom=composition_bottom,
                width=0.8,
                color=colour,
                edgecolor='white',
                linewidth=0.35,
            )
            composition_bottom += values

        annotation_years = set(
            rolling_rates.index[
                rolling_rates.index.to_series().mod(3).eq(0)
            ]
        )
        if not rolling_rates.empty:
            annotation_years.add(rolling_rates.index[-1])

        for year, rate_row in rolling_rates.iterrows():
            if year not in annotation_years:
                continue

            segment_bottom = 0
            for outcome_label in country_dashboard_colours:
                outcome_share = rate_row[outcome_label]
                if outcome_share >= 0.08:
                    ax_composition.text(
                        year,
                        segment_bottom + outcome_share / 2,
                        f'{outcome_share:.0%}',
                        ha='center',
                        va='center',
                        fontsize=8,
                        color=(
                            '#000000'
                            if outcome_label == 'Other protection'
                            else 'white'
                        ),
                            )
                segment_bottom += outcome_share

        ax_composition.set_ylim(0, 1)
        ax_composition.yaxis.set_major_formatter(mtick.PercentFormatter(1.0))
        ax_composition.set_xlabel('Calendar Year', fontsize=11)
        ax_composition.set_ylabel('Share of total decisions', fontsize=11)
        ax_composition.set_xticks(
            np.unique(
                np.append(
                    np.arange(2000, last_calendar_year + 1, 2),
                    last_calendar_year,
                )
            )
        )
        ax_composition.tick_params(axis='x', rotation=0)
        ax_composition.grid(axis='y', linestyle='--', alpha=0.35)
        ax_composition.set_axisbelow(True)
        for spine in ('top', 'right'):
            ax_composition.spines[spine].set_visible(False)

        country_label = country_dashboard_label(country_code)
        fig.suptitle(
            f'Asylum Applications, Decisions, and Outcomes: {country_label}, '
            f'2000–{last_calendar_year}',
            fontsize=15,
            y=0.995,
        )
        fig.text(
            0.5,
            0.955,
            (
                'Upper chart: annual decisions are stacked bars; '
                'applications are the blue dotted line.  '
                'Lower chart: up-to-3-year trailing outcome shares; '
                'windows with fewer than '
                f'{COUNTRY_DASHBOARD_MINIMUM_DECISIONS:,} decisions are omitted. '
                'UNHCR stock is shown only where loaded.'
            ),
            ha='center',
            fontsize=9,
            color='#4A235A',
        )

        legend_handles = [
            mpl.patches.Patch(
                color=colour,
                label=outcome_label,
            )
            for outcome_label, colour in country_dashboard_colours.items()
        ] + [applications_line]
        if stock_handle is not None:
            legend_handles.append(stock_handle)

        fig.legend(
            handles=legend_handles,
            loc='upper center',
            bbox_to_anchor=(0.5, 0.925),
            ncols=len(legend_handles),
            fontsize=9,
            frameon=False,
        )
        fig.tight_layout(rect=(0.03, 0.04, 0.99, 0.89))
        plt.show()

country_dashboard_selector.observe(
    render_country_dashboard,
    names='value',
)
dashboard_widget = widgets.VBox([
        widgets.HTML(
            '<b>Select a receiving country.</b> '
            'The dashboard updates while this notebook kernel is running.'
        ),
        country_dashboard_selector,
        country_dashboard_output,
    ])
dashboard_widget.add_class('unhcr-sans-serif-widget')
display(
    widgets.HTML(
        '<style>'
        '.unhcr-sans-serif-widget, '
        '.unhcr-sans-serif-widget * { '
        'font-family: Arial, Helvetica, sans-serif !important; '
        '}'
        '</style>'
    ),
    dashboard_widget,
)
render_country_dashboard()

Individual Countries - Applications by Origin for each Top 20 Receiving Countries

In [ ]:
# %% Applications by origin for the Top 20 asylum countries
# This keeps the UNHCR-reported application values. Case-to-person multipliers
# cannot be applied here because UNHCR says average case size varies by origin.
origin_country_names = iso_country_names.rename(columns={
    'ISO country code': 'Country of origin (ISO)',
    'Country name': 'Country of origin',
})
asylum_country_names = iso_country_names.rename(columns={
    'ISO country code': 'Country of asylum (ISO)',
    'Country name': 'Country of asylum',
})
top_20_destination_origin_applications = (
    UNapps.loc[
        UNapps['Year'].between(2000, last_calendar_year)
        & UNapps['Country of asylum (ISO)'].isin(top_20_worldwide_codes)
        ]
    .groupby(
        ['Year', 'Country of asylum (ISO)', 'Country of origin (ISO)'],
        as_index=False,
    )['applied']
    .sum()
    .rename(columns={'applied': 'Applications (reported)'})
    .merge(asylum_country_names, on='Country of asylum (ISO)', how='left')
    .merge(origin_country_names, on='Country of origin (ISO)', how='left')
)

# Some UNHCR aggregates or regional codes have no corresponding country name.
top_20_destination_origin_applications['Country of asylum'] = (
    top_20_destination_origin_applications['Country of asylum']
    .fillna(top_20_destination_origin_applications['Country of asylum (ISO)'])
)
top_20_destination_origin_applications['Country of origin'] = (
    top_20_destination_origin_applications['Country of origin']
    .fillna(top_20_destination_origin_applications['Country of origin (ISO)'])
)
top_20_destination_origin_applications = (
    top_20_destination_origin_applications
    .sort_values(
        ['Country of asylum (ISO)', 'Year', 'Applications (reported)'],
        ascending=[True, True, False],
    )
    .reset_index(drop=True)
)

Individual Countries - Create Application and Grant Rates by Origin for each Top 20 Receiving Country

In [ ]:
# %% Summarise application volumes and grant rates by origin and destination
# The Top 10 origin countries per destination are selected by application
# volume, then used in both charts below so the rate chart has the same scope.
top_20_destination_origin_decisions = (
    UNdecs.loc[
        UNdecs['Year'].between(2000, last_calendar_year)
        & UNdecs['Country of asylum (ISO)'].isin(top_20_worldwide_codes)
        ]
    .groupby(
        ['Country of asylum (ISO)', 'Country of origin (ISO)'],
        as_index=False,
    )[
        [
            'Recognized decisions',
            'Complementary protection',
            'Rejected decisions',
            'Otherwise closed',
            'Total decisions',
        ]
    ]
    .sum()
)
top_20_destination_origin_summary = (
    top_20_destination_origin_applications
    .groupby(
        [
            'Country of asylum (ISO)',
            'Country of asylum',
            'Country of origin (ISO)',
            'Country of origin',
        ],
        as_index=False,
    )['Applications (reported)']
    .sum()
    .merge(
        top_20_destination_origin_decisions,
        on=['Country of asylum (ISO)', 'Country of origin (ISO)'],
        how='left',
    )
)
decision_columns = [
    'Recognized decisions',
    'Complementary protection',
    'Rejected decisions',
    'Otherwise closed',
    'Total decisions',
]
top_20_destination_origin_summary[decision_columns] = (
    top_20_destination_origin_summary[decision_columns].fillna(0)
)
top_20_destination_origin_summary['Grant rate'] = np.where(
    top_20_destination_origin_summary['Total decisions'] > 0,
    top_20_destination_origin_summary['Recognized decisions']
    / top_20_destination_origin_summary['Total decisions'],
    np.nan,
)
for outcome_column, rate_column in {
    'Complementary protection': 'Other protection rate',
    'Rejected decisions': 'Refused rate',
    'Otherwise closed': 'Otherwise closed rate',
}.items():
    top_20_destination_origin_summary[rate_column] = np.where(
        top_20_destination_origin_summary['Total decisions'] > 0,
        top_20_destination_origin_summary[outcome_column]
        / top_20_destination_origin_summary['Total decisions'],
        np.nan,
    )
top_20_destination_origin_summary['Origin label'] = (
        top_20_destination_origin_summary['Country of origin']
        + ' (' + top_20_destination_origin_summary[
            'Country of origin (ISO)'] + ')'
)

top_origins_per_destination = (
    top_20_destination_origin_summary
    .sort_values(
        ['Country of asylum (ISO)', 'Applications (reported)'],
        ascending=[True, False],
    )
    .groupby('Country of asylum (ISO)', group_keys=False)
    .head(10)
    .copy()
)

Individual Countries - Plot Application Volumes and Grant Rates by Origin for each Top 20 Receiving Country in an Annotated Bar Chart

In [ ]:
# %% Individual Countries - Plot Applications by Origin for each Top 20 Receiving Country
facet_columns = 2
facet_rows = int(np.ceil(len(top_20_worldwide_codes) / facet_columns))

fig, axes = plt.subplots(
    facet_rows,
    facet_columns,
    figsize=(facet_columns * 7, facet_rows * 5.5),
)
facet_axes = np.atleast_1d(axes).ravel()

# Use a shared x-axis range so the country facets are directly comparable,
# with room for labels placed just outside shorter bars.
shared_x_limit = (
    top_origins_per_destination["Applications (reported)"].max() * 1.28
)

for ax, country_code in zip(facet_axes, top_20_worldwide_codes):
    origin_data = (
        top_origins_per_destination.loc[
            top_origins_per_destination["Country of asylum (ISO)"].eq(
                country_code
            )
        ]
        .sort_values("Applications (reported)")
        .reset_index(drop=True)
    )
    positions = np.arange(len(origin_data))

    application_bars = ax.barh(
        positions,
        origin_data["Applications (reported)"],
        color="#002664",
    )
    ax.set_title(
        country_code,
        fontsize=12,
        fontweight="regular",
        pad=10,
    )

    application_labels = ax.bar_label(
        application_bars,
        labels=[
            (
                f"{volume:,.0f}"
                + chr(10)
                + (
                    f"G {grant_rate:.0%}"
                    if pd.notna(grant_rate) else "G n/a"
                )
                + " | "
                + (
                    f"OP {other_protection_rate:.0%}"
                    if pd.notna(other_protection_rate) else "OP n/a"
                )
                + " | "
                + (
                    f"R {refused_rate:.0%}"
                    if pd.notna(refused_rate) else "R n/a"
                )
                + " | "
                + (
                    f"C {otherwise_closed_rate:.0%}"
                    if pd.notna(otherwise_closed_rate) else "C n/a"
                )
            )
            for (
                volume,
                grant_rate,
                other_protection_rate,
                refused_rate,
                otherwise_closed_rate,
            ) in zip(
                origin_data["Applications (reported)"],
                origin_data["Grant rate"],
                origin_data["Other protection rate"],
                origin_data["Refused rate"],
                origin_data["Otherwise closed rate"],
            )
        ],
        padding=3,
        fontsize=7,
        color="#000000",
    )

    # Move labels inside only where the bar is wide enough for both lines.
    long_bar_threshold = shared_x_limit * 0.45
    for label, value in zip(
            application_labels,
            origin_data["Applications (reported)"],
    ):
        if value >= long_bar_threshold:
            label.set_color("white")
            label.set_ha("right")
            label.set_position((-3, 0))

    ax.set_xlim(0, shared_x_limit)
    ax.xaxis.set_major_formatter(
        mtick.FuncFormatter(
            lambda value, _: format_volume_label(value)
        )
    )
    ax.set_yticks(positions)
    ax.set_yticklabels(origin_data["Country of origin (ISO)"])
    ax.tick_params(axis="y", labelsize=8, length=0)
    ax.tick_params(axis="x", labelsize=9)
    ax.grid(axis="x", linestyle="--", alpha=0.3)
    ax.set_axisbelow(True)

    for spine in ("top", "right", "left"):
        ax.spines[spine].set_visible(False)

for ax in facet_axes[len(top_20_worldwide_codes):]:
    ax.set_visible(False)

fig.suptitle(
    f"Application Volumes by Country of Origin: Top 20 Receiving Countries, "
    f"2000–{last_calendar_year}",
    fontsize=14,
    y=0.995,
)
fig.text(
    0.5,
    0.978,
    "G = Granted | OP = Other protection | R = Refused | C = Closed",
    ha="center",
    fontsize=10,
)
fig.tight_layout(rect=(0.04, 0.03, 1, 0.985))
plt.show()

Individual Countries - Plot Grant Rates by Origin for each Top 20 Receiving Country since 2000 in a Bar Chart

In [ ]:
# %% Individual Countries - Plot Grant Rates by Origin for each Top 20 Receiving Country
facet_columns = 2
fig, axes = plt.subplots(
    facet_rows, facet_columns,
    figsize=(facet_columns * 7, facet_rows * 5.5),
    sharex=False,
)
facet_axes = np.atleast_1d(axes).ravel()

for ax, country_code in zip(facet_axes, top_20_worldwide_codes):
    origin_data = top_origins_per_destination.loc[
        top_origins_per_destination['Country of asylum (ISO)'] == country_code
        ].sort_values('Applications (reported)')
    has_decisions = origin_data['Grant rate'].notna()
    bars = ax.barh(
        origin_data['Country of origin (ISO)'],
        origin_data['Grant rate'].fillna(0),
        color=np.where(has_decisions, '#00B050', '#7A7A7A'),
    )
    ax.bar_label(
        bars,
        labels=[
            f'{rate:.0%}' if pd.notna(rate) else 'n/a'
            for rate in origin_data['Grant rate']
        ],
        padding=3,
        fontsize=7,
    )
    ax.set_title(f'{country_code}',
                 fontsize=12, pad=5)
    ax.tick_params(axis='y', labelsize=7)
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(axis='x', linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    for spine in ('top', 'right', 'left'):
        ax.spines[spine].set_visible(False)

for ax in facet_axes[len(top_20_worldwide_codes):]:
    ax.set_visible(False)

for i in range(len(top_20_worldwide_codes)):
	facet_axes[i].set_xlim(0, 1)
	facet_axes[i].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))

fig.suptitle(
    f'Grant Rates by Country of Origin: Top 20 Receiving Countries, '
    f'2000–{last_calendar_year}',
    fontsize=14, y=0.985,
)
fig.supxlabel('Recognized decisions as a share of all decisions', fontsize=12,
              y=0.975)
fig.legend(
    handles=[
        mpl.patches.Patch(color='#00B050', label='Grant Rate')
    ],
    loc='upper center',
    bbox_to_anchor=(0.5, 0.975),
    ncols=2,
    frameon=False,
    fontsize=10,
)
fig.tight_layout(rect=(0.04, 0.03, 1, 0.97))
fig.subplots_adjust(hspace = 0.2, wspace = 0.2)
print('-' * 100)
plt.show()
print('-' * 100)

Last Calendar Year Analysis - Recent Applications and Decisions

In [ ]:
# %% Last Calendar Year - Origin and Destination Application Summary
last_calendar_year_destination_origin_decisions = (
    UNdecs.loc[
        (UNdecs['Year'] == last_calendar_year)
        & UNdecs['Country of asylum (ISO)'].isin(top_20_worldwide_codes)
        ]
    .groupby(
        ['Country of asylum (ISO)', 'Country of origin (ISO)'],
        as_index=False,
    )[
        [
            'Recognized decisions',
            'Complementary protection',
            'Rejected decisions',
            'Otherwise closed',
            'Total decisions',
        ]
    ]
    .sum()
)
last_calendar_year_destination_origin_summary = (
    UNapps.loc[
        (UNapps['Year'] == last_calendar_year)
        & UNapps['Country of asylum (ISO)'].isin(top_20_worldwide_codes)
        ]
    .groupby(
        ['Country of asylum (ISO)', 'Country of origin (ISO)'],
        as_index=False,
    )['applied']
    .sum()
    .rename(columns={'applied': 'Applications (reported)'})
    .merge(asylum_country_names, on='Country of asylum (ISO)', how='left')
    .merge(origin_country_names, on='Country of origin (ISO)', how='left')
    .merge(
        last_calendar_year_destination_origin_decisions,
        on=['Country of asylum (ISO)', 'Country of origin (ISO)'],
        how='left',
    )
)
last_calendar_year_destination_origin_summary[decision_columns] = (
    last_calendar_year_destination_origin_summary[decision_columns].fillna(0)
)
last_calendar_year_destination_origin_summary['Country of asylum'] = (
    last_calendar_year_destination_origin_summary['Country of asylum']
    .fillna(last_calendar_year_destination_origin_summary[
                'Country of asylum (ISO)'])
)
last_calendar_year_destination_origin_summary['Country of origin'] = (
    last_calendar_year_destination_origin_summary['Country of origin']
    .fillna(last_calendar_year_destination_origin_summary[
                'Country of origin (ISO)'])
)
last_calendar_year_destination_origin_summary['Grant rate'] = np.where(
    last_calendar_year_destination_origin_summary['Total decisions'] > 0,
    last_calendar_year_destination_origin_summary['Recognized decisions']
    / last_calendar_year_destination_origin_summary['Total decisions'],
    np.nan,
)
for outcome_column, rate_column in {
    'Complementary protection': 'Other protection rate',
    'Rejected decisions': 'Refused rate',
    'Otherwise closed': 'Otherwise closed rate',
}.items():
    last_calendar_year_destination_origin_summary[rate_column] = np.where(
        last_calendar_year_destination_origin_summary['Total decisions'] > 0,
        last_calendar_year_destination_origin_summary[outcome_column]
        / last_calendar_year_destination_origin_summary['Total decisions'],
        np.nan,
    )
last_calendar_year_destination_origin_summary['Origin label'] = (
        last_calendar_year_destination_origin_summary['Country of origin']
        + ' ('
        + last_calendar_year_destination_origin_summary[
            'Country of origin (ISO)']
        + ')'
)
last_calendar_year_top_origins_per_destination = (
    last_calendar_year_destination_origin_summary
    .sort_values(
        ['Country of asylum (ISO)', 'Applications (reported)'],
        ascending=[True, False],
    )
    .groupby('Country of asylum (ISO)', group_keys=False)
    .head(10)
    .copy()
)

Individual Countries - Last Calendar Year Application and Decisions for Top 20 Receiving Countries since 2000

In [ ]:
# %% Plot last-calendar-year application volumes by origin and destination
facet_columns = 2
fig, axes = plt.subplots(
    facet_rows, facet_columns,
    figsize=(facet_columns * 7, facet_rows * 5.5),
)
facet_axes = np.atleast_1d(axes).ravel()

for ax, country_code in zip(facet_axes, top_20_worldwide_codes):
    origin_data = last_calendar_year_top_origins_per_destination.loc[
        last_calendar_year_top_origins_per_destination[
            'Country of asylum (ISO)'] == country_code
        ].sort_values('Applications (reported)')
    bars = ax.barh(
        origin_data['Country of origin (ISO)'], origin_data['Applications (reported)'],
        color='#002664'
    )
    ax.bar_label(
        bars,
        labels=[
            (
                    f'{volume:,.0f}\n'
                    + (f'G {grant_rate:.0%}' if pd.notna(
                grant_rate) else 'G n/a')
                    + ' | '
                    + (
                        f'OP {other_protection_rate:.0%}'
                        if pd.notna(other_protection_rate) else 'OP n/a'
                    )
                    + ' | '
                    + (f'R {refused_rate:.0%}' if pd.notna(
                refused_rate) else 'R n/a')
                    + ' | '
                    + (
                        f'C {otherwise_closed_rate:.0%}'
                        if pd.notna(otherwise_closed_rate) else 'C n/a'
                    )
            )
            for (
                volume, grant_rate, other_protection_rate, refused_rate,
                otherwise_closed_rate
            ) in zip(
                origin_data['Applications (reported)'],
                origin_data['Grant rate'],
                origin_data['Other protection rate'],
                origin_data['Refused rate'],
                origin_data['Otherwise closed rate'],
            )
        ],
        padding=3,
        fontsize=7,
    )
    ax.margins(x=0.35)
    ax.xaxis.set_major_formatter(EngFormatter(unit = '', places = 0))
    ax.set_title(f'{country_code}', fontsize=12, pad=5)
    ax.tick_params(axis='y', labelsize=7)
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(axis='x', linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    for spine in ('top', 'right', 'left'):
        ax.spines[spine].set_visible(False)

for ax in facet_axes[len(top_20_worldwide_codes):]:
    ax.set_visible(False)

fig.suptitle(
    f'Top 10 Application Volumes (with grant rates) by Country of Origin: Top 20 Receiving Countries since 2000, '
    f'{last_calendar_year}',
    fontsize=14, x = 0.55, y=0.99,
)
fig.supxlabel(
    'G = Granted | OP = Other protection | R = Refused | C = Otherwise closed',
    fontsize=10, x = 0.55, y=0.98,
)
fig.tight_layout(rect=(0.1, 0.03, 1, 0.98))
#fig.subplots_adjust(hspace = 0.2, wspace = 0.2)
print('-' * 100)
plt.show()
print('-' * 100)

Individual Countries - Plot Last Calendar Year Grant Rates by Origin for each Top 20 Receiving Country

In [ ]:
# %% Plot last-calendar-year grant rates by origin and destination
facet_columns = 2
fig, axes = plt.subplots(
    facet_rows, facet_columns,
    figsize=(facet_columns * 7, facet_rows * 5.5),
    sharex=False,
)
facet_axes = np.atleast_1d(axes).ravel()

for ax, country_code in zip(facet_axes, top_20_worldwide_codes):
    origin_data = last_calendar_year_top_origins_per_destination.loc[
        last_calendar_year_top_origins_per_destination[
            'Country of asylum (ISO)'] == country_code
        ].sort_values('Applications (reported)')
    has_decisions = origin_data['Grant rate'].notna()
    bars = ax.barh(
        origin_data['Country of origin (ISO)'],
        origin_data['Grant rate'].fillna(0),
        color=np.where(has_decisions, '#00B050', '#7A7A7A'),
    )
    ax.bar_label(
        bars,
        labels=[
            f'{rate:.0%}' if pd.notna(rate) else 'n/a'
            for rate in origin_data['Grant rate']
        ],
        padding=3,
        fontsize=7,
    )
    ax.set_title(f'{country_code}',
                 fontsize=12, pad=5)
    ax.tick_params(axis='y', labelsize=7)
    ax.tick_params(axis='x', labelsize=8)
    ax.grid(axis='x', linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)
    for spine in ('top', 'right', 'left'):
        ax.spines[spine].set_visible(False)

for ax in facet_axes[len(top_20_worldwide_codes):]:
    ax.set_visible(False)

for i in range(len(top_20_worldwide_codes)):
	facet_axes[i].set_xlim(0, 1)
	facet_axes[i].xaxis.set_major_formatter(mtick.PercentFormatter(1.0))

fig.suptitle(
    f'Grant Rates for Top 10 Countries of Origin: Top 20 Receiving Countries since 2000, {last_calendar_year}',
    fontsize=14, x = 0.55, y=0.995,
)
fig.supxlabel('Percent of Applications Granted', fontsize=12, x = 0.55, y=0.985)
fig.legend(
    handles=[
        mpl.patches.Patch(color='#00B050', label='Grant Rate')
    ],
    loc='upper center',
    bbox_to_anchor=(0.55, 0.98),
    ncols=2,
    frameon=False,
    fontsize=10,
)
fig.tight_layout(rect=(0.12, 0.06, 1, 0.975))
#fig.subplots_adjust(hspace=0.2, wspace = 0.1)
print('-' * 100)
plt.show()
print('-' * 100)

In [ ]:
# %% Individual Countries - Last Calendar Year Applications and Grant Rates
# %% Compare last-calendar-year applications and grant rates by applicant
# origin
# Application volume and grant rate use separate x-axes because they are
# different measures; each origin has one bar for each measure.
facet_columns = 2
fig, axes = plt.subplots(
    facet_rows, facet_columns,
    figsize=(facet_columns * 8, facet_rows * 5.8),
)
facet_axes = np.atleast_1d(axes).ravel()

for ax, country_code in zip(facet_axes, top_20_worldwide_codes):
    origin_data = last_calendar_year_top_origins_per_destination.loc[
        last_calendar_year_top_origins_per_destination[
            'Country of asylum (ISO)'] == country_code
        ].sort_values('Applications (reported)')
    positions = np.arange(len(origin_data))
    application_bars = ax.barh(
        positions - 0.2,
        origin_data['Applications (reported)'],
        height=0.38,
        color='#002664',
        label='Applications',
    )
    ax.bar_label(
        application_bars,
        labels=[f'{value:,.0f}' for value in
                origin_data['Applications (reported)']],
        padding=3,
        fontsize=6,
        color='#002664',
    )
    ax.margins(x=0.25)
    ax.xaxis.set_major_formatter(EngFormatter(unit = '', places = 0))
    ax.set_yticks(positions, origin_data['Country of origin (ISO)'])
    ax.tick_params(axis='y', labelsize=7)
    ax.tick_params(axis='x', labelsize=8, colors='#002664')
    ax.grid(axis='x', linestyle='--', alpha=0.3)
    ax.set_axisbelow(True)

    rate_axis = ax.twiny() # Second axis.
    has_decisions = origin_data['Grant rate'].notna()
    rate_bars = rate_axis.barh(
        positions + 0.2,
        origin_data['Grant rate'].fillna(0),
        height=0.38,
        color=np.where(has_decisions, '#00B050', '#7A7A7A'),
        label='Grant rate',
    )
    rate_axis.bar_label(
        rate_bars,
        labels=[
            f'{rate:.0%}' if pd.notna(rate) else 'n/a'
            for rate in origin_data['Grant rate']
        ],
        padding=3,
        fontsize=6,
        color='#00B050',
    )
    rate_axis.set_xlim(0, 1)
    rate_axis.xaxis.set_major_formatter(mtick.PercentFormatter(1.0))
    rate_axis.tick_params(axis='x', labelsize=8, colors='#00B050')
    rate_axis.spines['top'].set_color('#00B050')

    ax.set_title(f'{country_code}', fontsize=12,
                 pad=22)
    for spine in ('top', 'right', 'left'):
        ax.spines[spine].set_visible(False)
    rate_axis.spines['right'].set_visible(False)
    rate_axis.spines['left'].set_visible(False)

for ax in facet_axes[len(top_20_worldwide_codes):]:
    ax.set_visible(False)

fig.legend(
    handles=[
        mpl.patches.Patch(color='#002664', label='Applications (bottom '
                                                 'axis)'),
        mpl.patches.Patch(color='#00B050', label='Grant rate (top axis)'),
    ],
    loc='upper center', bbox_to_anchor=(0.572, 0.985), ncols=3, frameon=False,
)
fig.suptitle(
    f'Applications and Grant Rates for Top 10 Countries of Origin: Top 20 Receiving Countries since 2000, '
    f'{last_calendar_year}',
    fontsize=14, x = .572, y=0.995,
)
fig.tight_layout(rect=(0.12, 0, 1, 0.98))
fig.subplots_adjust(hspace = 0.2, wspace = 0.1)
print('-' * 100)
plt.show()
print('-' * 100)

Countries of Origin - Create Top 20 Countries of Origin by applications since 2000

In [ ]:
top_20_origin_countries = (
    UNapps.loc[
        UNapps['Year'].between(2000, last_calendar_year)
    ]
    .groupby('Country of origin (ISO)', as_index=False)['applied']
    .sum()
    .rename(
        columns={
            'Country of origin (ISO)': 'ISO country code',
            'applied': 'Total applications',
        }
    )
    .nlargest(20, 'Total applications')
    .reset_index(drop=True)
)

top_20_origin_countries_display = (
    top_20_origin_countries
    .merge(
        iso_country_names.set_index('ISO country code')[['Country name']],
        left_on='ISO country code',
        right_index=True,
        how='left',
    )
    [['ISO country code', 'Country name', 'Total applications']]
)


display(
    top_20_origin_countries_display.style
    .format({'Total applications': '{:,.0f}'})
    .hide(axis='index')
)

In [ ]:
# Assign one persistent colour to every receiving-country ISO code that
# appears in the full dataset, so interactive origin-country charts retain
# the same colour meaning as the Top 20 charts.
all_destination_codes = sorted(
    pd.concat(
        [
            UNapps.loc[
                UNapps['Year'].between(2000, last_calendar_year),
                'Country of asylum (ISO)',
            ],
            UNdecs.loc[
                UNdecs['Year'].between(2000, last_calendar_year),
                'Country of asylum (ISO)',
            ],
        ]
    )
    .dropna()
    .astype(str)
    .loc[lambda codes: codes.str.fullmatch(r'[A-Z]{3}')]
    .unique()
)

global_colour_palette = plt.get_cmap(
    'turbo',
    len(all_destination_codes),
)

destination_colour_map = {
    country_code: mpl.colors.to_hex(global_colour_palette(index))
    for index, country_code in enumerate(all_destination_codes)
}
destination_colour_map['OTHER'] = '#7A7A7A'

In [ ]:
def plot_origin_application_grant_comparison(
    nationality_code,
    top_destination_count=6,
):
    applications = UNapps.loc[
        UNapps['Year'].between(2000, last_calendar_year)
        & UNapps['Country of origin (ISO)'].eq(nationality_code)
    ].copy()

    grants = UNdecs.loc[
        UNdecs['Year'].between(2000, last_calendar_year)
        & UNdecs['Country of origin (ISO)'].eq(nationality_code)
    ].copy()

    if applications.empty:
        print(f'No application data available for {nationality_code}.')
        return

    application_totals = (
        applications
        .groupby('Country of asylum (ISO)')['applied']
        .sum()
        .rename('Applications')
    )

    grant_totals = (
        grants
        .groupby('Country of asylum (ISO)')['Recognized decisions']
        .sum()
        .rename('Asylum Granted')
    )

    destination_importance = pd.concat(
        [application_totals, grant_totals],
        axis=1,
    ).fillna(0)

    application_total = destination_importance['Applications'].sum()
    grant_total = destination_importance['Asylum Granted'].sum()

    destination_importance['Importance'] = (
        destination_importance['Applications']
        .div(application_total if application_total else 1)
        + destination_importance['Asylum Granted']
        .div(grant_total if grant_total else 1)
    )

    comparison_destinations = (
        destination_importance
        .nlargest(top_destination_count, 'Importance')
        .index
        .tolist()
    )

    destination_groups = comparison_destinations + ['OTHER']
    comparison_years = range(2000, last_calendar_year + 1)

    def annual_destination_shares(data, value_column):
        data = data.copy()

        data['Destination group'] = np.where(
            data['Country of asylum (ISO)'].isin(comparison_destinations),
            data['Country of asylum (ISO)'],
            'OTHER',
        )

        annual_volumes = (
            data
            .groupby(['Year', 'Destination group'])[value_column]
            .sum()
            .unstack(fill_value=0)
            .reindex(
                index=comparison_years,
                columns=destination_groups,
                fill_value=0,
            )
        )

        return (
            annual_volumes
            .div(annual_volumes.sum(axis=1), axis=0)
            .fillna(0)
            * 100
        )

    application_shares = annual_destination_shares(
        applications,
        'applied',
    )
    grant_shares = annual_destination_shares(
        grants,
        'Recognized decisions',
    )
    share_difference = grant_shares - application_shares

    # Keep receiving-country colours consistent across every origin-country
    # chart.  The shared map is based on ISO code, never a chart-local rank.
    destination_colours = {
        destination: destination_colour_map[destination]
        for destination in comparison_destinations
    }
    destination_colours['OTHER'] = destination_colour_map['OTHER']

    def destination_code(destination):
        return 'OTH' if destination == 'OTHER' else destination

    def annotate_stack_end(ax, shares, minimum_share=4):
        final_year = shares.index[-1]
        cumulative_share = 0

        for destination in destination_groups:
            share = shares.loc[final_year, destination]

            if share >= minimum_share:
                ax.text(
                    final_year - 0.3,
                    cumulative_share + share / 2,
                    destination_code(destination),
                    ha='right',
                    va='center',
                    fontsize=8,
                    fontweight='bold',
                    color='white',
                )

            cumulative_share += share

    fig, axes = plt.subplots(
        3,
        1,
        figsize=(16, 15),
        sharex=True,
    )

    # Applications panel
    axes[0].stackplot(
        application_shares.index,
        application_shares.T,
        colors=[
            destination_colours[destination]
            for destination in destination_groups
        ],
    )
    annotate_stack_end(axes[0], application_shares)
    axes[0].set_title('Applications made:', loc='left', fontsize=13)
    axes[0].set_ylabel('Share of Applications')
    axes[0].set_ylim(0, 100)

    # Recognised-grants panel
    axes[1].stackplot(
        grant_shares.index,
        grant_shares.T,
        colors=[
            destination_colours[destination]
            for destination in destination_groups
        ],
    )
    annotate_stack_end(axes[1], grant_shares)
    axes[1].set_title(
        'Asylum Granted:',
        loc='left',
        fontsize=13,
    )
    axes[1].set_ylabel('Share of Grants')
    axes[1].set_ylim(0, 100)

    # Difference panel
    axes[2].axhline(0, color='#000000', linewidth=0.8)

    positive_bottom = np.zeros(len(share_difference))
    negative_bottom = np.zeros(len(share_difference))

    for destination in destination_groups:
        difference = share_difference[destination].to_numpy()
        positive_values = np.clip(difference, 0, None)
        negative_values = np.clip(difference, None, 0)

        positive_bars = axes[2].bar(
            share_difference.index,
            positive_values,
            bottom=positive_bottom,
            width=0.85,
            color=destination_colours[destination],
            edgecolor='white',
            linewidth=0.3,
        )

        negative_bars = axes[2].bar(
            share_difference.index,
            negative_values,
            bottom=negative_bottom,
            width=0.85,
            color=destination_colours[destination],
            edgecolor='white',
            linewidth=0.3,
        )

        axes[2].bar_label(
            positive_bars,
            labels=[
                destination_code(destination) if value >= 2.5 else ''
                for value in positive_values
            ],
            label_type='center',
            fontsize=6,
            color='white',
        )

        axes[2].bar_label(
            negative_bars,
            labels=[
                destination_code(destination) if value <= -2.5 else ''
                for value in negative_values
            ],
            label_type='center',
            fontsize=6,
            color='white',
        )

        positive_bottom += positive_values
        negative_bottom += negative_values

    difference_limit = share_difference.abs().to_numpy().max()
    difference_limit = max(5, np.ceil(difference_limit / 5) * 5)

    axes[2].set_ylim(-difference_limit, difference_limit)
    axes[2].set_title(
        'Difference: Grant share minus Application share',
        loc='left',
        fontsize=13,
    )
    axes[2].set_xlabel('Calendar Year')
    axes[2].set_ylabel('Percentage-point Difference')

    for ax in axes:
        ax.yaxis.set_major_formatter(mtick.PercentFormatter(100))
        ax.grid(axis='y', linestyle='--', alpha=0.3)
        ax.set_axisbelow(True)

        for spine in ('top', 'right'):
            ax.spines[spine].set_visible(False)

    legend_handles = [
        mpl.patches.Patch(
            color=destination_colours[destination],
            label='OTH' if destination == 'OTHER' else destination,
        )
        for destination in destination_groups
    ]

    fig.suptitle(
        f'Applications and Grants by Receiving Country. Country of Origin: '
        f'{country_label(nationality_code)}, '
        f'2000–{last_calendar_year}',
        fontsize=16,
        y=0.99,
    )

    fig.legend(
        handles=legend_handles,
        title='Receiving country:',
        loc='upper center',
        bbox_to_anchor=(0.5, 0.962),
        ncols=10,
        frameon=False,
    )

    fig.tight_layout(rect=(0, 0, 1, 0.92))
    plt.show()

In [ ]:
# %% Plot the Top 20 Origin Countries and their destinations
for nationality_code in top_20_origin_countries['ISO country code']:
    plot_origin_application_grant_comparison(nationality_code)

In [ ]:
# %% Interactive origin-country receiving-country comparison
# Requires an active Jupyter kernel: choose an origin country to redraw the
# same applications, grants, and share-difference charts as above.

origin_country_coverage = (
    UNapps.loc[
        UNapps['Year'].between(2000, last_calendar_year)
    ]
    .groupby('Country of origin (ISO)')
    .agg(
        reported_application_years=('Year', 'nunique'),
        reported_applications=('applied', 'sum'),
    )
)
origin_country_coverage = origin_country_coverage.loc[
    origin_country_coverage.index.to_series()
    .astype(str)
    .str.fullmatch(r'[A-Z]{3}')
    & origin_country_coverage['reported_applications'].gt(0)
].copy()
origin_country_codes = sorted(origin_country_coverage.index)

def origin_country_dashboard_label(country_code):
    country = pycountry.countries.get(alpha_3=country_code)
    country_name = country.name if country is not None else country_code
    reported_years = origin_country_coverage.loc[
        country_code, 'reported_application_years'
    ]
    return (
        f'{country_code} — {country_name} '
        f'({reported_years} reported application years)'
    )

origin_country_selector = widgets.Dropdown(
    options=[
        (origin_country_dashboard_label(country_code), country_code)
        for country_code in origin_country_codes
    ],
    value='AFG' if 'AFG' in origin_country_codes else origin_country_codes[0],
    description='Country of origin:',
    style={'description_width': 'initial'},
    layout=widgets.Layout(width='650px'),
)
origin_country_output = widgets.Output()

def render_origin_country_dashboard(change=None):
    with origin_country_output:
        clear_output(wait=True)
        plot_origin_application_grant_comparison(
            origin_country_selector.value
        )

origin_country_selector.observe(
    render_origin_country_dashboard,
    names='value',
)
dashboard_widget = widgets.VBox([
        widgets.HTML(
            '<b>Select a country of origin.</b> '
            'Only origins with reported application data are listed.  '
            'The charts update while this notebook kernel is running.'
        ),
        origin_country_selector,
        origin_country_output,
    ])
dashboard_widget.add_class('unhcr-sans-serif-widget')
display(
    widgets.HTML(
        '<style>'
        '.unhcr-sans-serif-widget, '
        '.unhcr-sans-serif-widget * { '
        'font-family: Arial, Helvetica, sans-serif !important; '
        '}'
        '</style>'
    ),
    dashboard_widget,
)
render_origin_country_dashboard()